# DFFM-VSM: Complete CIFER Fraud Detection Reproduction Notebook

**Purpose:** reproduce the workflow reported in `Fraud_Detection_FINAL.pdf` as faithfully as the manuscript specifies, while separating **manuscript-reported numbers** from **new numbers actually produced by this notebook**.

The manuscript specifies: 64 selected features → 8×8 grayscale image; ViT (4 layers, 4 heads, 64 dim), Swin (2 stages, 2×2 windows, 64→128), Mamba (2 layers, state dim 16, 64 dim, 20 past transactions), GAN fraud synthesis on the training split only, DFFM adaptive fusion, Adam, batch size 32, learning rate 1e-4, 100 epochs, chronological 70/15/15 splitting, CNN/LSTM baselines, 8 ablation configurations, 5-fold chronological CV, SHAP/Grad-CAM, leakage checks, and cross-dataset evaluation.

## Important reproduction audit

The current public Hugging Face CIFER dataset page describes a 21-million-row dataset with **11 visible raw columns** (`step`, `type`, `amount`, `nameOrig`, `oldbalanceOrg`, `newbalanceOrig`, `nameDest`, `oldbalanceDest`, `newbalanceDest`, `isFraud`, `isFlaggedFraud`). The manuscript, however, states that CIFER has **434 raw features** including merchant category, device type, location, and payment method. Those two descriptions do not match. Therefore this notebook includes a **64-feature engineering compatibility layer** from the currently public raw CIFER schema. This is necessary for a runnable experiment, but it is not evidence that the manuscript's unstated 434-feature upstream construction has been exactly recovered.

Likewise, the manuscript does not fully specify the GAN target ratio, exact feature quotas, exact feature-to-pixel ordering, exact unseen-fraud definition, exact cross-dataset preprocessing, or exact fold construction. Those are explicitly configurable below and are recorded in the run manifest.

**Never copy manuscript-reported values into the empirical result table.** The notebook reports them separately as a reference table.

## Manuscript-reported reference results

The uploaded manuscript reports the following values. They are stored only as a **reference target**, not as generated measurements.

- Proposed CIFER result: Accuracy 97.84%, Precision 96.93%, Recall 96.23%, F1 96.57%, ROC-AUC 0.9816, PR-AUC 0.979.
- Plain three-branch concatenation: Accuracy 97.34%, PR-AUC 0.975.
- DFFM improvement over concatenation: +0.50 percentage points Accuracy and +0.004 PR-AUC.
- 5-fold chronological CV: Accuracy 97.58 ± 0.27%, F1 0.964 ± 0.003, ROC-AUC 0.978 ± 0.002, PR-AUC 0.966 ± 0.002.
- Ablation table values are reproduced below in code.

The manuscript itself also contains an internal inconsistency: a paragraph says the DFFM FPR is 2.80%, while Table IV gives FPR 2.10% and specificity 0.979. Since `FPR = 1 - Specificity`, the table's 2.10% is internally consistent; the notebook flags this discrepancy rather than silently resolving it as an experimental fact.

In [1]:
# ============================================================
# 0. PAPER-REPORTED REFERENCE VALUES (NOT EXPERIMENTAL OUTPUT)
# ============================================================
import os, json, math, gc, random, time, warnings, hashlib, subprocess, sys, platform
from pathlib import Path
import numpy as np
import pandas as pd

MANUSCRIPT_ABLATION = pd.DataFrame([
    ["ViT only", 94.62, 0.944, 0.862, 0.926, 4.80, 0.901, 0.952],
    ["Swin only", 95.13, 0.951, 0.841, 0.914, 5.90, 0.887, 0.941],
    ["Mamba only", 95.48, 0.956, 0.850, 0.920, 5.40, 0.894, 0.946],
    ["ViT + Swin", 96.24, 0.964, 0.889, 0.941, 3.90, 0.921, 0.961],
    ["ViT + Mamba", 96.71, 0.969, 0.901, 0.951, 3.30, 0.934, 0.967],
    ["Swin + Mamba", 97.02, 0.972, 0.883, 0.938, 4.20, 0.917, 0.958],
    ["ViT + Swin + Mamba (concat.)", 97.34, 0.975, np.nan, np.nan, np.nan, np.nan, np.nan],
    ["DFFM-VSM (proposed)", 97.84, 0.979, 0.932, 0.968, 2.10, 0.958, 0.979],
], columns=["Model","Accuracy_pct","PR_AUC","MCC","Balanced_Accuracy","FPR_pct","Recall","Specificity"])

MANUSCRIPT_BASELINE = pd.DataFrame([
    ["CNN", 91.2, 89.4, 87.1, 88.2, 0.89, 0.86],
    ["LSTM", 93.5, 91.3, 89.8, 90.5, 0.91, 0.88],
    ["DFFM-VSM (proposed)", 97.84, 96.93, 96.23, 96.57, 0.9816, 0.979],
], columns=["Model","Accuracy_pct","Precision_pct","Recall_pct","F1_pct","ROC_AUC","PR_AUC"])

MANUSCRIPT_CV = pd.DataFrame([
    ["Fold-1",97.21,0.961,0.975,0.962],
    ["Fold-2",97.63,0.964,0.979,0.966],
    ["Fold-3",97.91,0.968,0.981,0.969],
    ["Fold-4",97.48,0.963,0.977,0.964],
    ["Fold-5",97.67,0.966,0.980,0.968],
], columns=["Fold","Accuracy_pct","F1","ROC_AUC","PR_AUC"])

MANUSCRIPT_CROSSDATA = pd.DataFrame([
    ["CIFER Fraud",97.84,96.93,96.23,96.57,0.9816],
    ["IEEE-CIS Fraud",96.91,95.87,95.11,95.48,0.9718],
    ["PaySim",96.24,95.02,94.73,94.87,0.9632],
    ["European Credit Card",95.78,94.45,93.89,94.16,0.9514],
    ["Synthetic Stress-Test",94.85,93.70,92.88,93.50,0.9470],
], columns=["Dataset","Accuracy_pct","Precision_pct","Recall_pct","F1_pct","ROC_AUC"])

MANUSCRIPT_SHAP = pd.DataFrame([
    ["Transaction Amount",0.92,"Very High"],
    ["Transaction Frequency",0.88,"Very High"],
    ["Merchant Category",0.81,"High"],
    ["Timestamp Pattern",0.79,"High"],
    ["Device Type",0.72,"Medium"],
    ["Payment Method",0.68,"Medium"],
], columns=["Feature","SHAP_Importance","Contribution_Level"])

REPORT = {
    "proposed": {
        "Accuracy": 0.9784,
        "Precision": 0.9693,
        "Recall": 0.9623,
        "F1": 0.9657,
        "ROC_AUC": 0.9816,
        "PR_AUC": 0.9790,
    },
    "dffm_gain_accuracy_points": 0.50,
    "dffm_gain_pr_auc": 0.004,
    "cv_accuracy_mean": 0.9758,
    "cv_accuracy_std": 0.0027,
    "reported_dffm_fpr_paragraph_pct": 2.80,
    "reported_dffm_fpr_table_pct": 2.10,
}

print(MANUSCRIPT_ABLATION.to_string(index=False))

                       Model  Accuracy_pct  PR_AUC   MCC  Balanced_Accuracy  FPR_pct  Recall  Specificity
                    ViT only         94.62   0.944 0.862              0.926      4.8   0.901        0.952
                   Swin only         95.13   0.951 0.841              0.914      5.9   0.887        0.941
                  Mamba only         95.48   0.956 0.850              0.920      5.4   0.894        0.946
                  ViT + Swin         96.24   0.964 0.889              0.941      3.9   0.921        0.961
                 ViT + Mamba         96.71   0.969 0.901              0.951      3.3   0.934        0.967
                Swin + Mamba         97.02   0.972 0.883              0.938      4.2   0.917        0.958
ViT + Swin + Mamba (concat.)         97.34   0.975   NaN                NaN      NaN     NaN          NaN
         DFFM-VSM (proposed)         97.84   0.979 0.932              0.968      2.1   0.958        0.979


## 0B. Manuscript-reported supporting tables

These are transcribed from the uploaded manuscript for audit/reproduction only. They are kept separate from measurements produced by this notebook.

In [2]:
# ============================================================
# 0B. MANUSCRIPT-REPORTED SUPPORTING TABLES (REFERENCE ONLY)
# ============================================================

MANUSCRIPT_TECHNIQUES = pd.DataFrame([
    ["Vision Transformer (ViT)", "Global dependency learning", "Q=XWQ, K=XWK, V=XWV; scaled dot-product self-attention"],
    ["Swin Transformer", "Local and hierarchical feature extraction", "Window-based attention, shifted windows, relative position bias"],
    ["Mamba (State Space Model)", "Sequential behavioral modeling", "x(t+1)=Ax(t)+Bu(t), y(t)=Cx(t)+Du(t)"],
    ["GAN", "Synthetic fraud generation", "min_G max_D V(D,G)"],
    ["Cross-Entropy Loss", "Fraud classification loss", "L=-(1/N) sum y_i log(y_hat_i)"],
    ["DFFM", "Adaptive global/local/temporal fusion", "F_final=alpha F_ViT+beta F_Swin+gamma F_Mamba; alpha+beta+gamma=1"],
    ["SHAP + Grad-CAM", "Decision explanation", "Feature attribution and gradient-based localization"],
    ["Adam", "Optimization", "Adaptive moment estimation"],
], columns=["Technique","Purpose","Mathematical_Concept"])

MANUSCRIPT_LEAKAGE = pd.DataFrame([
    ["CIFER Fraud Dataset","Yes","Yes","Yes","Yes","Very Low","Valid"],
    ["IEEE CIS Fraud Dataset","Yes","Yes","Yes","Partial","Low","Valid"],
    ["PaySim Dataset","Yes","Yes","Yes","Yes","Very Low","Highly Valid"],
    ["European Credit Card Dataset","Yes","Yes","Yes","Partial","Low","Valid"],
    ["Synthetic Stress-Test Dataset","Yes","Yes","Yes","Yes","Very Low","Highly Valid"],
], columns=["Dataset","GAN_Train_Only","Chronological_Split","Transaction_Overlap_Removed","Unseen_Fraud_Test_Cases","Leakage_Risk","Final_Evaluation_Validity"])

MANUSCRIPT_CAPABILITY = pd.DataFrame([
    ["Roy et al.","2018","","Yes","",""],
    ["Jurgovsky et al.","2018","","Yes","",""],
    ["Cheng et al. (STAN)","2020","","Yes","Yes",""],
    ["Dastidar et al.","2022","","Yes","",""],
    ["Fiore et al.","2019","Yes","","",""],
    ["Ghaleb et al.","2023","Yes","","",""],
    ["Mienye and Sun","2023","Yes","","",""],
    ["Jiang et al.","2023","","","Yes",""],
    ["Huang et al.","2020","","","Yes",""],
    ["Gorishniy et al.","2021","","","Yes",""],
    ["Alharbi et al.","2022","","","","Yes"],
    ["DFFM-VSM (proposed)","2026","Yes","Yes","Yes","Yes"],
], columns=["Method","Year","Imbal.","Seq.","Attn.","Tab-to-img"])

MANUSCRIPT_CONFIG = pd.DataFrame([
    ["Image size / patch size","8x8 / 2x2 (16 patches)"],
    ["ViT depth / heads / embedding","4 / 4 / 64"],
    ["Swin stages / window / embedding","2 / 2x2 / 64->128"],
    ["Mamba state dim. / layers / embedding","16 / 2 / 64"],
    ["Sequence length L","20 past transactions"],
    ["GAN generator / discriminator","3-layer MLP, 128 hidden units"],
    ["Classifier head","FC(256)->FC(128)->FC(64), dropout 0.3"],
    ["Optimizer","Adam, beta1=0.9, beta2=0.999"],
    ["Learning rate / batch size / epochs","1e-4 / 32 / 100"],
    ["Random seed","42"],
], columns=["Component","Value"])

for name, table in {
    "manuscript_reported_techniques.csv": MANUSCRIPT_TECHNIQUES,
    "manuscript_reported_leakage.csv": MANUSCRIPT_LEAKAGE,
    "manuscript_reported_capability.csv": MANUSCRIPT_CAPABILITY,
    "manuscript_reported_configuration.csv": MANUSCRIPT_CONFIG,
}.items():
    table.to_csv(OUTPUT_DIR / name, index=False)

print("Technique table:")
print(MANUSCRIPT_TECHNIQUES.to_string(index=False))
print("\nLeakage table:")
print(MANUSCRIPT_LEAKAGE.to_string(index=False))
print("\nCapability table:")
print(MANUSCRIPT_CAPABILITY.to_string(index=False))
print("\nReference configuration:")
print(MANUSCRIPT_CONFIG.to_string(index=False))


NameError: name 'OUTPUT_DIR' is not defined

In [3]:
!pip -q install -U pandas numpy scikit-learn matplotlib seaborn tqdm joblib shap huggingface_hub pyarrow


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.5/828.5 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 101.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the 

In [ ]:
# ============================================================
# 1. INSTALL DEPENDENCIES
# ============================================================

# Core packages

# Official Mamba installation. The current upstream repository recommends
# installing against the already-installed Colab PyTorch using --no-build-isolation.
# https://github.com/state-spaces/mamba
import subprocess, sys

def _pip_install(args):
    # Removed -q (quiet) flag to get verbose output for debugging
    cmd=[sys.executable, "-m", "pip", "install", "--upgrade", "--no-build-isolation"] + args
    print("Running:", " ".join(cmd))
    return subprocess.call(cmd)

_mamba_rc = _pip_install(["mamba-ssm[causal-conv1d]"])
if _mamba_rc != 0:
    print("Primary Mamba install failed; retrying the core package without extras...")
    _mamba_rc = _pip_install(["mamba-ssm"])

print("Mamba pip return code:", _mamba_rc)

Running: /usr/bin/python3 -m pip install --upgrade --no-build-isolation mamba-ssm[causal-conv1d]


In [ ]:
import warnings
warnings.filterwarnings("ignore", message="The numpy.array_api module is experimental") # Suppress specific numpy warning

import platform # Added for platform.python_version()

# ============================================================
# 2. IMPORTS, DEVICE, REPRODUCIBILITY, OUTPUT DIRECTORIES
# ============================================================

# Fix for numpy/scipy/scikit-learn compatibility on Python 3.13 in Colab.
# Ensure latest numpy (2.x) compatible with Python 3.13 is installed, then install compatible scipy/scikit-learn.
!pip uninstall scipy scikit-learn numpy -y
!pip install --upgrade --force-reinstall --ignore-installed numpy
!pip install --upgrade --force-reinstall scipy
!pip install --upgrade --force-reinstall scikit-learn


In [ ]:
import os
import gc
import json
import math
import time
import random
import warnings
import hashlib
from pathlib import Path
from dataclasses import dataclass, asdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from tqdm.auto import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, matthews_corrcoef,
    balanced_accuracy_score, confusion_matrix, roc_curve,
    precision_recall_curve
)
from sklearn.model_selection import TimeSeriesSplit
import joblib

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

warnings.filterwarnings("ignore")

In [ ]:

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OUTPUT_DIR = Path("/content/dffm_vsm_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

## 3. Run profiles

The paper's reference configuration says **100 epochs / batch 32 / Adam / 1e-4**, but a full 21M-row, 8-model, 5-fold experiment is far beyond a normal free Colab session.

- `COLAB` is the runnable verification profile and samples the public dataset across all 14 parts.
- `PAPER` sets the manuscript hyperparameters and requests all 14 parts. You should use a high-RAM/high-GPU runtime and expect a long run.
- `AUDIT_ONLY` runs the preprocessing audit and manuscript-reference figures without training.

The notebook records the active profile in `run_manifest.json`.

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="The numpy.array_api module is experimental") # Suppress specific numpy warning

import platform # Added for platform.python_version()

# ============================================================
# 3. CONFIGURATION
# ============================================================

RUN_PROFILE = "COLAB"   # "COLAB", "PAPER", or "AUDIT_ONLY"

# Architecture fidelity
REQUIRE_OFFICIAL_MAMBA = False
USE_GAN = True
USE_MIXED_PRECISION = True

# Main CIFER source
CIFER_REPO = "CiferAI/Cifer-Fraud-Detection-Dataset-AF"
NUM_PARTS = 14
MAX_ROWS_PER_PART = 100_000      # COLAB profile; PAPER sets this to None

# Data split
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Feature engineering / selection
NUM_SELECTED_FEATURES = 64
MI_SAMPLE_SIZE = 50_000
CLIP_LOWER = 0.005
CLIP_UPPER = 0.995

# Sequence model
SEQUENCE_LENGTH = 20
SEQUENCE_GROUP_PRIMARY = "nameOrig"

# GAN: the manuscript does not specify an exact target ratio.
# We use a configurable train-only multiplier, recorded as an assumption.
GAN_FRAUD_MULTIPLIER = 1.0
GAN_MAX_SYNTHETIC = 100_000
GAN_EPOCHS = 15
GAN_BATCH_SIZE = 256
GAN_LATENT_DIM = 32
GAN_LR = 2e-4
GAN_NOISE_STD = 0.01

# Training
BATCH_SIZE = 32
EPOCHS = 100
LR = 1e-4
WEIGHT_DECAY = 0.0
PATIENCE = 10
NUM_WORKERS = 2
THRESHOLD = 0.50
GRAD_CLIP = 1.0

# Evaluation toggles
RUN_BASELINES = True
RUN_ABLATIONS = True
RUN_CV = True
RUN_XAI = True
RUN_EXTERNAL_DATASETS = False
RUN_STRESS_TEST = True

# To avoid unexpectedly huge runs in Colab, these are the defaults in COLAB mode.
if RUN_PROFILE == "COLAB":
    NUM_PARTS = 14
    MAX_ROWS_PER_PART = 25_000
    BATCH_SIZE = 64
    EPOCHS = 5
    PATIENCE = 3
    RUN_CV = False
    RUN_EXTERNAL_DATASETS = False
    RUN_XAI = True
elif RUN_PROFILE == "PAPER":
    NUM_PARTS = 14
    MAX_ROWS_PER_PART = None
    BATCH_SIZE = 32
    EPOCHS = 100
    PATIENCE = 10
    RUN_CV = True
    RUN_EXTERNAL_DATASETS = True
    RUN_XAI = True
elif RUN_PROFILE == "AUDIT_ONLY":
    RUN_ABLATIONS = False
    RUN_BASELINES = False
    RUN_CV = False
    RUN_XAI = False
    RUN_EXTERNAL_DATASETS = False
    RUN_STRESS_TEST = False

CONFIG = {
    "run_profile": RUN_PROFILE,
    "cifer_repo": CIFER_REPO,
    "num_parts": NUM_PARTS,
    "max_rows_per_part": MAX_ROWS_PER_PART,
    "image_size": "8x8",
    "patch_size": "2x2",
    "vit_depth": 4,
    "vit_heads": 4,
    "vit_embed_dim": 64,
    "swin_window": 2,
    "swin_stage_dims": [64,128],
    "mamba_state_dim": 16,
    "mamba_layers": 2,
    "mamba_embed_dim": 64,
    "sequence_length": SEQUENCE_LENGTH,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LR,
    "optimizer": "Adam",
    "betas": [0.9, 0.999],
    "seed": SEED,
    "gan_enabled": USE_GAN,
    "gan_multiplier": GAN_FRAUD_MULTIPLIER,
    "gan_target_ratio_exact": None,
    "threshold": THRESHOLD,
    "notes": [
        "Current public CIFER page exposes 11 raw columns; manuscript states 434 raw features.",
        "64-feature compatibility layer is therefore engineered here; it is not silently assumed to equal the manuscript's unstated 434-feature pipeline.",
        "GAN oversampling ratio was not specified in the manuscript and is configurable.",
        "Exact cross-dataset preprocessing and unseen-fraud definition were not fully specified."
    ]
}

with open(OUTPUT_DIR / "run_manifest.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

print(json.dumps(CONFIG, indent=2))

In [ ]:
# ============================================================
# 4. DOWNLOAD PUBLIC CIFER FILES
# ============================================================

from huggingface_hub import hf_hub_download

cifer_dir = Path("/content/cifer_data")
cifer_dir.mkdir(exist_ok=True)

cifer_files = []
for i in range(1, NUM_PARTS + 1):
    filename = f"Cifer-Fraud-Detection-Dataset-AF-part-{i}-14.csv"
    local_path = cifer_dir / filename
    if local_path.exists():
        print("Already present:", local_path)
    else:
        print("Downloading:", filename)
        downloaded = hf_hub_download(
            repo_id=CIFER_REPO,
            filename=filename,
            repo_type="dataset",
            local_dir=str(cifer_dir)
        )
        local_path = Path(downloaded)
    cifer_files.append(local_path)

print("\nFiles:")
for p in cifer_files:
    print(p)

In [ ]:
# ============================================================
# 5. LOAD CIFER RAW DATA
# ============================================================

USE_COLUMNS = [
    "step", "type", "amount", "nameOrig", "oldbalanceOrg",
    "newbalanceOrig", "nameDest", "oldbalanceDest",
    "newbalanceDest", "isFraud", "isFlaggedFraud"
]

parts = []
for p in cifer_files:
    print("Reading", p.name)
    if MAX_ROWS_PER_PART is None:
        part = pd.read_csv(p, usecols=lambda c: c in USE_COLUMNS)
    else:
        part = pd.read_csv(p, usecols=lambda c: c in USE_COLUMNS, nrows=MAX_ROWS_PER_PART)
    parts.append(part)

raw_df = pd.concat(parts, ignore_index=True)
del parts
gc.collect()

print("Raw shape:", raw_df.shape)
print(raw_df.dtypes)
print(raw_df.head())
print("\nClass distribution:")
print(raw_df["isFraud"].value_counts(dropna=False))

In [ ]:
# ============================================================
# 5B. PUBLIC CIFER SCHEMA AUDIT
# ============================================================

public_schema = list(raw_df.columns)
expected_public_schema = [
    "step", "type", "amount", "nameOrig", "oldbalanceOrg",
    "newbalanceOrig", "nameDest", "oldbalanceDest", "newbalanceDest",
    "isFraud", "isFlaggedFraud"
]

source_audit = {
    "public_columns_loaded": public_schema,
    "expected_current_public_columns": expected_public_schema,
    "public_column_count": int(len(public_schema)),
    "manuscript_claimed_raw_feature_count": 434,
    "schema_matches_manuscript_434_feature_claim": len(public_schema) == 434,
    "note": "The current public CIFER schema is used as the executable source; the manuscript's 434-feature upstream representation is not publicly specified in sufficient detail to reconstruct exactly."
}

with open(OUTPUT_DIR/"source_schema_audit.json", "w") as f:
    json.dump(source_audit, f, indent=2)

print(json.dumps(source_audit, indent=2))


In [ ]:
# ============================================================
# 6. DATA CLEANING + CHRONOLOGICAL ORDER + OVERLAP FINGERPRINT
# ============================================================

def clean_cifer(df):
    df = df.copy()
    for c in ["step", "amount", "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest", "isFraud", "isFlaggedFraud"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    if "type" in df.columns:
        df["type"] = df["type"].fillna("UNKNOWN").astype(str)
    for c in ["nameOrig", "nameDest"]:
        if c in df.columns:
            df[c] = df[c].fillna("UNKNOWN").astype(str)
    df = df.drop_duplicates().reset_index(drop=True)
    df = df.sort_values("step", kind="stable").reset_index(drop=True)
    return df

raw_df = clean_cifer(raw_df)

FINGERPRINT_COLS = [
    c for c in ["step","type","amount","nameOrig","nameDest",
                "oldbalanceOrg","newbalanceOrig","oldbalanceDest","newbalanceDest"]
    if c in raw_df.columns
]

def transaction_fingerprint_frame(df):
    return (
        df[FINGERPRINT_COLS].astype(str).agg("|".join, axis=1)
    )

raw_df["_fingerprint"] = transaction_fingerprint_frame(raw_df)

print("Cleaned shape:", raw_df.shape)
print("Time range:", raw_df["step"].min(), "to", raw_df["step"].max())

## 7. Feature engineering compatibility layer

The manuscript says 434 features were reduced to 64 using domain aggregates + mutual information. The current public CIFER page exposes only 11 raw columns, so this notebook constructs a richer candidate feature set from those public columns using the current transaction and **strictly past-only** account/entity history. No global sender/receiver frequency is computed because that would leak future/test information. Candidates are clipped, imputed, standardized, mutual-information ranked on the training split, and reduced to exactly 64 features.

Feature groups used for the 8x8 layout:

- amount/balance
- time
- entity/history surrogate
- transaction/type
- behavioral

There is no current public `merchant category`, `device type`, `payment method`, or `location` column in the public 11-column schema. The notebook therefore labels entity-history features as **surrogates** instead of pretending those variables exist.

In [ ]:
# ============================================================
# 7. FEATURE ENGINEERING
# ============================================================

def _safe_div(a, b):
    return a / (b + 1e-6)


def build_candidate_features(df):
    df = df.copy().sort_values("step", kind="stable").reset_index(drop=True)

    # Required columns for generic handling
    for c, default in {
        "amount": 0.0,
        "step": 0.0,
        "oldbalanceOrg": 0.0,
        "newbalanceOrig": 0.0,
        "oldbalanceDest": 0.0,
        "newbalanceDest": 0.0,
        "isFlaggedFraud": 0,
        "nameOrig": "UNKNOWN",
        "nameDest": "UNKNOWN",
        "type": "UNKNOWN",
    }.items():
        if c not in df.columns:
            df[c] = default

    features = {}
    groups = {}

    def add(name, values, group):
        features[name] = pd.to_numeric(values, errors="coerce").astype("float32")
        groups[name] = group

    # --------------------------------------------------------
    # Base amount / balance features
    # --------------------------------------------------------
    amount = df["amount"].astype(float)
    old_o = df["oldbalanceOrg"].astype(float)
    new_o = df["newbalanceOrig"].astype(float)
    old_d = df["oldbalanceDest"].astype(float)
    new_d = df["newbalanceDest"].astype(float)

    add("amount", amount, "amount_balance")
    add("log_amount", np.log1p(np.clip(amount, 0, None)), "amount_balance")
    add("amount_sqrt", np.sqrt(np.clip(amount, 0, None)), "amount_balance")
    add("amount_sq", amount**2, "amount_balance")
    add("oldbalanceOrg", old_o, "amount_balance")
    add("newbalanceOrig", new_o, "amount_balance")
    add("oldbalanceDest", old_d, "amount_balance")
    add("newbalanceDest", new_d, "amount_balance")
    add("orig_balance_change", old_o - new_o, "amount_balance")
    add("dest_balance_change", new_d - old_d, "amount_balance")
    add("orig_error", new_o - (old_o - amount), "amount_balance")
    add("dest_error", new_d - (old_d + amount), "amount_balance")
    add("amount_to_orig_before", _safe_div(amount, old_o), "amount_balance")
    add("amount_to_dest_before", _safe_div(amount, old_d), "amount_balance")
    add("orig_after_zero", (new_o == 0).astype(float), "amount_balance")
    add("orig_before_zero", (old_o == 0).astype(float), "amount_balance")
    add("dest_after_zero", (new_d == 0).astype(float), "amount_balance")
    add("dest_before_zero", (old_d == 0).astype(float), "amount_balance")
    add("net_balance_delta", (new_o - old_o) + (new_d - old_d), "amount_balance")
    add("total_balance_before", old_o + old_d, "amount_balance")

    # Additional deterministic, current-transaction features. These use only
    # the current row and therefore do not introduce future-information leakage.
    total_before = old_o + old_d
    total_after = new_o + new_d
    add("amount_log10", np.log10(1.0 + np.clip(amount, 0, None)), "amount_balance")
    add("amount_cbrt", np.cbrt(np.clip(amount, 0, None)), "amount_balance")
    add("amount_abs", np.abs(amount), "amount_balance")
    add("amount_to_total_balance_before", _safe_div(amount, total_before), "amount_balance")
    add("orig_remaining_ratio", _safe_div(new_o, old_o), "amount_balance")
    add("dest_remaining_ratio", _safe_div(new_d, old_d), "amount_balance")
    add("orig_depletion_fraction", _safe_div(np.maximum(old_o - new_o, 0), old_o), "amount_balance")
    add("dest_growth_fraction", _safe_div(np.maximum(new_d - old_d, 0), old_d), "amount_balance")
    add("balance_asymmetry", _safe_div(old_o - old_d, total_before), "amount_balance")
    add("total_balance_after", total_after, "amount_balance")
    add("total_balance_change", total_after - total_before, "amount_balance")
    add("absolute_balance_error", np.abs(new_o - (old_o - amount)) + np.abs(new_d - (old_d + amount)), "amount_balance")

    # --------------------------------------------------------
    # Time features
    # --------------------------------------------------------
    step = df["step"].astype(float)
    hour = np.mod(step, 24.0)
    day = np.floor(step / 24.0)
    add("step", step, "time")
    add("hour", hour, "time")
    add("day", day, "time")
    add("hour_sin", np.sin(2*np.pi*hour/24), "time")
    add("hour_cos", np.cos(2*np.pi*hour/24), "time")
    add("day_sin", np.sin(2*np.pi*day/31), "time")
    add("day_cos", np.cos(2*np.pi*day/31), "time")
    add("day_of_week", np.mod(day, 7), "time")
    add("is_weekend", (np.mod(day, 7) >= 5).astype(float), "time")
    add("hour_fraction", hour / 24.0, "time")

    # --------------------------------------------------------
    # Transaction type features
    # --------------------------------------------------------
    type_series = df["type"].fillna("UNKNOWN").astype(str)
    for t in sorted(type_series.unique()):
        safe_t = str(t).replace(" ", "_").replace("-", "_")
        is_t = (type_series == t).astype(float)
        add(f"type_{safe_t}", is_t, "transaction_type")
        add(f"amount_x_type_{safe_t}", amount * is_t, "transaction_type")

    # --------------------------------------------------------
    # Entity-history / behavioral features, all based on current + past only
    # --------------------------------------------------------
    orig = df["nameOrig"].fillna("UNKNOWN").astype(str)
    dest = df["nameDest"].fillna("UNKNOWN").astype(str)

    # sender count before current
    orig_count_before = orig.groupby(orig, sort=False).cumcount().astype(float)
    dest_count_before = dest.groupby(dest, sort=False).cumcount().astype(float)
    add("sender_count_before", orig_count_before, "entity_surrogate")
    add("receiver_count_before", dest_count_before, "entity_surrogate")
    add("sender_is_new", (orig_count_before == 0).astype(float), "entity_surrogate")
    add("receiver_is_new", (dest_count_before == 0).astype(float), "entity_surrogate")

    orig_prev_step = step.groupby(orig, sort=False).shift(1)
    dest_prev_step = step.groupby(dest, sort=False).shift(1)
    sender_gap = (step - orig_prev_step).fillna(0)
    receiver_gap = (step - dest_prev_step).fillna(0)
    add("sender_time_gap", sender_gap, "entity_surrogate")
    add("receiver_time_gap", receiver_gap, "entity_surrogate")
    add("sender_gap_inverse", _safe_div(1.0, 1.0 + sender_gap), "entity_surrogate")
    add("receiver_gap_inverse", _safe_div(1.0, 1.0 + receiver_gap), "entity_surrogate")

    orig_sum_before = amount.groupby(orig, sort=False).cumsum() - amount
    dest_sum_before = amount.groupby(dest, sort=False).cumsum() - amount
    orig_mean_before = _safe_div(orig_sum_before, orig_count_before.clip(lower=1))
    dest_mean_before = _safe_div(dest_sum_before, dest_count_before.clip(lower=1))
    add("sender_amount_mean_before", orig_mean_before, "behavior")
    add("receiver_amount_mean_before", dest_mean_before, "behavior")

    orig_sq_sum_before = (amount**2).groupby(orig, sort=False).cumsum() - amount**2
    dest_sq_sum_before = (amount**2).groupby(dest, sort=False).cumsum() - amount**2
    orig_var_before = np.maximum(_safe_div(orig_sq_sum_before, orig_count_before.clip(lower=1)) - orig_mean_before**2, 0)
    dest_var_before = np.maximum(_safe_div(dest_sq_sum_before, dest_count_before.clip(lower=1)) - dest_mean_before**2, 0)
    add("sender_amount_std_before", np.sqrt(orig_var_before), "behavior")
    add("receiver_amount_std_before", np.sqrt(dest_var_before), "behavior")
    add("sender_amount_vs_mean", _safe_div(amount, orig_mean_before.replace(0, np.nan)).replace([np.inf,-np.inf], np.nan).fillna(0), "behavior")
    add("receiver_amount_vs_mean", _safe_div(amount, dest_mean_before.replace(0, np.nan)).replace([np.inf,-np.inf], np.nan).fillna(0), "behavior")

    # sender and receiver type counts before current
    for t in sorted(type_series.unique()):
        safe_t = str(t).replace(" ", "_").replace("-", "_")
        is_t = (type_series == t).astype(float)
        sender_t_before = is_t.groupby(orig, sort=False).cumsum() - is_t
        receiver_t_before = is_t.groupby(dest, sort=False).cumsum() - is_t
        add(f"sender_typecount_{safe_t}", sender_t_before, "behavior")
        add(f"receiver_typecount_{safe_t}", receiver_t_before, "behavior")


    # --------------------------------------------------------
    # Generic numerical features for cross-dataset compatibility.
    # Target is excluded to prevent label leakage. Highly unique integer IDs
    # are skipped when their uniqueness ratio is > 0.995.
    # --------------------------------------------------------
    for c in df.columns:
        if c in {"isFraud", "type", "nameOrig", "nameDest"}:
            continue
        if not pd.api.types.is_numeric_dtype(df[c]):
            continue
        vals = pd.to_numeric(df[c], errors="coerce")
        if vals.notna().sum() == 0:
            continue
        uniqueness = vals.nunique(dropna=True) / max(1, vals.notna().sum())
        if uniqueness > 0.995 and str(c).lower().endswith("id"):
            continue
        lname = str(c).lower()
        if any(k in lname for k in ["time", "step", "timestamp", "dt", "date"]):
            grp = "time"
        elif any(k in lname for k in ["amount", "balance", "amt", "price", "fee"]):
            grp = "amount_balance"
        elif any(k in lname for k in ["device", "merchant", "category", "card", "location", "country"]):
            grp = "entity_surrogate"
        elif any(k in lname for k in ["flag", "fraud", "risk", "count", "freq", "velocity"]):
            grp = "behavior"
        else:
            grp = "behavior"
        safe_name = "raw_" + str(c).replace(" ", "_").replace("-", "_")
        if safe_name not in features:
            add(safe_name, vals, grp)

    # Do not compute global sender/receiver frequency here: that would use future
    # rows from validation/test and would violate the manuscript's leakage-aware rule.
    # Entity information is represented only by past-only features above.

    out = pd.DataFrame(features)
    return out, groups

candidate_df, candidate_groups = build_candidate_features(raw_df)
print("Candidate feature count:", candidate_df.shape[1])
print("Group counts:", pd.Series(candidate_groups).value_counts().to_dict())
print(candidate_df.head())

In [ ]:
# ============================================================
# 8. TEMPORAL 70/15/15 SPLIT + EXACT OVERLAP REMOVAL
# ============================================================

n = len(raw_df)
train_end = int(n * TRAIN_RATIO)
val_end = int(n * (TRAIN_RATIO + VAL_RATIO))

split_labels = np.empty(n, dtype=object)
split_labels[:train_end] = "train"
split_labels[train_end:val_end] = "val"
split_labels[val_end:] = "test"

# Exact transaction overlap removal based on the public fields available.
train_fp = set(raw_df.loc[:train_end-1, "_fingerprint"].astype(str))
val_keep = ~raw_df.loc[train_end:val_end-1, "_fingerprint"].isin(train_fp).to_numpy()
test_keep = ~raw_df.loc[val_end:, "_fingerprint"].isin(train_fp).to_numpy()

val_idx = np.arange(train_end, val_end)[val_keep]
test_idx = np.arange(val_end, n)[test_keep]
train_idx = np.arange(0, train_end)

print("Train rows:", len(train_idx))
print("Validation rows after overlap filtering:", len(val_idx))
print("Test rows after overlap filtering:", len(test_idx))

leakage_audit = {
    "train_rows": int(len(train_idx)),
    "val_rows_after_exact_train_overlap_removal": int(len(val_idx)),
    "test_rows_after_exact_train_overlap_removal": int(len(test_idx)),
    "train_test_exact_fingerprint_intersection": int(len(train_fp.intersection(set(raw_df.iloc[test_idx]["_fingerprint"].astype(str))))),
    "time_min_train": float(raw_df.iloc[train_idx]["step"].min()),
    "time_max_train": float(raw_df.iloc[train_idx]["step"].max()),
    "time_min_val": float(raw_df.iloc[val_idx]["step"].min()),
    "time_max_val": float(raw_df.iloc[val_idx]["step"].max()),
    "time_min_test": float(raw_df.iloc[test_idx]["step"].min()),
    "time_max_test": float(raw_df.iloc[test_idx]["step"].max()),
}

with open(OUTPUT_DIR / "leakage_overlap_audit.json", "w") as f:
    json.dump(leakage_audit, f, indent=2)

print(json.dumps(leakage_audit, indent=2))

In [ ]:
# ============================================================
# 9. LEAKAGE-SAFE IMPUTATION, CLIPPING, MUTUAL INFORMATION, 64-FEATURE SELECTION
# ============================================================

# Candidate data aligned to raw_df indices
Xcand = candidate_df.reset_index(drop=True)
y_all = raw_df["isFraud"].astype(int).to_numpy()

X_train_cand = Xcand.iloc[train_idx].copy()
X_val_cand = Xcand.iloc[val_idx].copy()
X_test_cand = Xcand.iloc[test_idx].copy()

y_train = y_all[train_idx]
y_val = y_all[val_idx]
y_test = y_all[test_idx]

# 1) Imputation fit only on train
imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_train_cand)
X_val_imp = imputer.transform(X_val_cand)
X_test_imp = imputer.transform(X_test_cand)

# 2) Quantile clipping thresholds fit only on train
lower_q = np.nanquantile(X_train_imp, CLIP_LOWER, axis=0)
upper_q = np.nanquantile(X_train_imp, CLIP_UPPER, axis=0)

X_train_clip = np.clip(X_train_imp, lower_q, upper_q)
X_val_clip = np.clip(X_val_imp, lower_q, upper_q)
X_test_clip = np.clip(X_test_imp, lower_q, upper_q)

# 3) Standardization fit only on train
scaler_all = StandardScaler()
X_train_scaled_all = scaler_all.fit_transform(X_train_clip).astype(np.float32)
X_val_scaled_all = scaler_all.transform(X_val_clip).astype(np.float32)
X_test_scaled_all = scaler_all.transform(X_test_clip).astype(np.float32)

# 4) MI ranking uses training samples only
mi_rng = np.random.default_rng(SEED)
mi_n = min(MI_SAMPLE_SIZE, len(X_train_scaled_all))
mi_idx = mi_rng.choice(len(X_train_scaled_all), size=mi_n, replace=False)

mi = mutual_info_classif(
    X_train_scaled_all[mi_idx],
    y_train[mi_idx],
    random_state=SEED,
    discrete_features=False
)

feature_names_all = list(Xcand.columns)
mi_scores = pd.Series(mi, index=feature_names_all).sort_values(ascending=False)

# Group-aware selection: retain representation from all manuscript-specified groups,
# then fill the remaining slots globally by MI score.
quota = {
    "amount_balance": 18,
    "time": 10,
    "entity_surrogate": 8,
    "transaction_type": 8,
    "behavior": 20,
}

selected = []
for group_name, q in quota.items():
    names = [n for n in feature_names_all if candidate_groups.get(n) == group_name]
    names = sorted(names, key=lambda n: mi_scores.get(n, -np.inf), reverse=True)
    selected.extend(names[:q])

selected = list(dict.fromkeys(selected))
remaining = [n for n in mi_scores.index if n not in selected]
selected.extend(remaining[:max(0, NUM_SELECTED_FEATURES - len(selected))])
selected = selected[:NUM_SELECTED_FEATURES]

if len(selected) < NUM_SELECTED_FEATURES:
    raise RuntimeError(f"Could not select {NUM_SELECTED_FEATURES} features; only {len(selected)} candidates available.")

selected_mi = mi_scores.loc[selected].sort_values(ascending=False)

selected_indices = [feature_names_all.index(n) for n in selected]
X_train64 = X_train_scaled_all[:, selected_indices].astype(np.float32)
X_val64 = X_val_scaled_all[:, selected_indices].astype(np.float32)
X_test64 = X_test_scaled_all[:, selected_indices].astype(np.float32)

feature_groups_64 = {n: candidate_groups[n] for n in selected}

selection_df = pd.DataFrame({
    "feature": selected,
    "group": [feature_groups_64[n] for n in selected],
    "mutual_information": [mi_scores[n] for n in selected]
})
selection_df.to_csv(OUTPUT_DIR / "selected_64_features.csv", index=False)

joblib.dump({
    "imputer": imputer,
    "lower_q": lower_q,
    "upper_q": upper_q,
    "scaler": scaler_all,
    "feature_names": selected,
    "feature_groups": feature_groups_64,
}, OUTPUT_DIR / "preprocessing.joblib")

print("Selected exactly", X_train64.shape[1], "features")
print(selection_df.head(20).to_string(index=False))

In [ ]:
# ============================================================
# 10. FIXED 8×8 TABULAR-TO-IMAGE MAPPING
# ============================================================

assert X_train64.shape[1] == 64

IMAGE_SIZE = 8

X_train_img = X_train64.reshape(-1, 1, IMAGE_SIZE, IMAGE_SIZE)
X_val_img = X_val64.reshape(-1, 1, IMAGE_SIZE, IMAGE_SIZE)
X_test_img = X_test64.reshape(-1, 1, IMAGE_SIZE, IMAGE_SIZE)

# Save exact mapping order
mapping_df = selection_df.copy()
mapping_df["pixel_index"] = np.arange(64)
mapping_df["row"] = mapping_df["pixel_index"] // 8
mapping_df["col"] = mapping_df["pixel_index"] % 8
mapping_df.to_csv(OUTPUT_DIR / "tabular_to_image_feature_mapping.csv", index=False)

print("Image shape:", X_train_img.shape)

In [ ]:
# ============================================================
# 11. CORRELATION MATRIX
# ============================================================

corr_sample_n = min(10_000, len(X_train64))
rng = np.random.default_rng(SEED)
cidx = rng.choice(len(X_train64), size=corr_sample_n, replace=False)

corr = pd.DataFrame(
    X_train64[cidx],
    columns=selected
).corr()

plt.figure(figsize=(15, 12))
sns.heatmap(corr, cmap="coolwarm", center=0, xticklabels=False, yticklabels=False)
plt.title("Feature Correlation Matrix — Selected 64 CIFER Features")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "feature_correlation_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

## 12. History construction for Mamba/LSTM

The manuscript specifies **20 past transactions for the same account or card**, with chronological ordering, padding, and masking. The public CIFER schema contains `nameOrig`, so this notebook uses the sender as the available account surrogate.

History windows are built from transactions that occurred **before** the current transaction. Test rows may use earlier training transactions as legitimate past context; no future transaction and no future label is used.

In [ ]:
# ============================================================
# 12. BUILD PAST-ONLY SEQUENCE INDICES (L=20)
# ============================================================

def build_history_indices(group_ids, seq_len=20):
    group_ids = np.asarray(group_ids)
    n = len(group_ids)
    hist = np.full((n, seq_len), -1, dtype=np.int32)

    # Stable group order preserves chronological order because raw_df is already time sorted.
    groups = pd.Series(group_ids).groupby(group_ids, sort=False).indices
    for _, idxs in groups.items():
        idxs = np.asarray(idxs, dtype=np.int32)
        if len(idxs) <= 1:
            continue
        m = min(seq_len, len(idxs) - 1)
        for k in range(1, m + 1):
            hist[idxs[k:], -k] = idxs[:-k]
    return hist

# Use sender account; if absent in external adapters, a global sequence is used there.
sequence_group = raw_df[SEQUENCE_GROUP_PRIMARY].fillna("UNKNOWN").astype(str).to_numpy()
history_idx_full = build_history_indices(sequence_group, SEQUENCE_LENGTH)

# Remove rows filtered for exact overlap in val/test by selecting the same positions.
train_hist_idx = history_idx_full[train_idx]
val_hist_idx = history_idx_full[val_idx]
test_hist_idx = history_idx_full[test_idx]

# Full transformed feature matrix in raw_df row space. All transforms were fit on train only.
X_all_imp = imputer.transform(Xcand)
X_all_clip = np.clip(X_all_imp, lower_q, upper_q)
X_all_scaled = scaler_all.transform(X_all_clip).astype(np.float32)
X64_full_for_history = X_all_scaled[:, selected_indices].astype(np.float32)

print("History matrix train:", train_hist_idx.shape)
print("History matrix val:", val_hist_idx.shape)
print("History matrix test:", test_hist_idx.shape)

In [ ]:
# ============================================================
# 13. MATERIALIZE SEQUENCE TENSORS FOR THE ACTIVE DATASET
# ============================================================

def materialize_history(history_indices, full_features, current_features):
    n = len(history_indices)
    d = full_features.shape[1]
    out = np.zeros((n, history_indices.shape[1], d), dtype=np.float32)
    for row in range(n):
        ids = history_indices[row]
        valid = ids >= 0
        if np.any(valid):
            out[row, valid] = full_features[ids[valid]]
        # If no history, keep zero padding. The current transaction is not copied into the past.
    return out

# Current transaction feature vectors are also retained for optional synthetic sequence handling.
X_train_seq = materialize_history(train_hist_idx, X64_full_for_history, X_train64)
X_val_seq = materialize_history(val_hist_idx, X64_full_for_history, X_val64)
X_test_seq = materialize_history(test_hist_idx, X64_full_for_history, X_test64)

print("Train sequence tensor:", X_train_seq.shape)

## 14. Train-only GAN fraud synthesis

The manuscript specifies a GAN trained **separately on training fraud samples only**. It does not specify an exact oversampling target. This notebook therefore uses the configurable `GAN_FRAUD_MULTIPLIER=1.0` by default: generated fraud count equals the number of real training fraud examples, capped by `GAN_MAX_SYNTHETIC`.

Synthetic rows are appended only to the training split. Validation and test remain real/unseen. For GAN-generated transactions, no genuine account history exists, so their Mamba history is zero-padded; this is recorded as an implementation limitation rather than hidden.

In [ ]:
# ============================================================
# 14. TRAIN-ONLY GAN
# ============================================================

class FraudGenerator(nn.Module):
    def __init__(self, latent_dim=32, out_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, out_dim),
        )
    def forward(self, z):
        return self.net(z)

class FraudDiscriminator(nn.Module):
    def __init__(self, in_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 1),
            nn.Sigmoid(),
        )
    def forward(self, x):
        return self.net(x)


def train_fraud_gan(X_fraud, epochs=15, batch_size=256, latent_dim=32, lr=2e-4):
    if len(X_fraud) < 32:
        print("Too few fraud samples for GAN; GAN augmentation skipped.")
        return None, None

    G = FraudGenerator(latent_dim, X_fraud.shape[1]).to(DEVICE)
    D = FraudDiscriminator(X_fraud.shape[1]).to(DEVICE)
    opt_g = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))
    bce = nn.BCELoss()

    X_t = torch.tensor(X_fraud, dtype=torch.float32)
    from torch.utils.data import TensorDataset
    loader = DataLoader(TensorDataset(X_t), batch_size=batch_size, shuffle=True, drop_last=False)

    for ep in range(epochs):
        d_losses, g_losses = [], []
        for (real,) in loader:
            real = real.to(DEVICE)
            bs = real.size(0)
            ones = torch.ones(bs,1,device=DEVICE)
            zeros = torch.zeros(bs,1,device=DEVICE)

            # D
            z = torch.randn(bs, latent_dim, device=DEVICE)
            fake = G(z).detach()
            opt_d.zero_grad(set_to_none=True)
            d_loss = bce(D(real), ones) + bce(D(fake), zeros)
            d_loss.backward()
            opt_d.step()

            # G
            z = torch.randn(bs, latent_dim, device=DEVICE)
            opt_g.zero_grad(set_to_none=True)
            fake = G(z)
            g_loss = bce(D(fake), ones)
            g_loss.backward()
            opt_g.step()

            d_losses.append(d_loss.item())
            g_losses.append(g_loss.item())

        if (ep + 1) % 5 == 0 or ep == 0:
            print(f"GAN epoch {ep+1:02d}/{epochs} | D={np.mean(d_losses):.4f} | G={np.mean(g_losses):.4f}")

    return G, D


if USE_GAN and RUN_PROFILE != "AUDIT_ONLY":
    fraud_train = X_train64[y_train == 1]
    G, D = train_fraud_gan(
        fraud_train,
        epochs=GAN_EPOCHS,
        batch_size=GAN_BATCH_SIZE,
        latent_dim=GAN_LATENT_DIM,
        lr=GAN_LR
    )

    n_to_generate = min(
        int(len(fraud_train) * GAN_FRAUD_MULTIPLIER),
        GAN_MAX_SYNTHETIC
    )

    if G is not None and n_to_generate > 0:
        synth_parts = []
        with torch.no_grad():
            for start in range(0, n_to_generate, 4096):
                n_batch = min(4096, n_to_generate - start)
                z = torch.randn(n_batch, GAN_LATENT_DIM, device=DEVICE)
                synth_parts.append(G(z).cpu().numpy())
        X_synth = np.vstack(synth_parts).astype(np.float32)
        if GAN_NOISE_STD > 0:
            X_synth += np.random.normal(0, GAN_NOISE_STD, X_synth.shape).astype(np.float32)
    else:
        X_synth = np.empty((0, 64), dtype=np.float32)
else:
    X_synth = np.empty((0, 64), dtype=np.float32)

print("Synthetic fraud generated:", len(X_synth))

## 15. Dataset objects

In [ ]:
# ============================================================
# 15. DATASET / DATALOADER
# ============================================================

class HybridFraudDataset(Dataset):
    def __init__(self, images, sequences, labels):
        self.images = torch.tensor(images, dtype=torch.float32)
        self.sequences = torch.tensor(sequences, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return self.images[idx], self.sequences[idx], self.labels[idx]

# Build training arrays with train-only GAN fraud synthesis.
if len(X_synth) > 0:
    synth_images = X_synth.reshape(-1, 1, 8, 8)
    synth_sequences = np.zeros((len(X_synth), SEQUENCE_LENGTH, 64), dtype=np.float32)
    X_train_final_img = np.concatenate([X_train_img, synth_images], axis=0)
    X_train_final_seq = np.concatenate([X_train_seq, synth_sequences], axis=0)
    y_train_final = np.concatenate([y_train, np.ones(len(X_synth), dtype=np.int64)], axis=0)
else:
    X_train_final_img = X_train_img
    X_train_final_seq = X_train_seq
    y_train_final = y_train

train_ds = HybridFraudDataset(X_train_final_img, X_train_final_seq, y_train_final)
val_ds = HybridFraudDataset(X_val_img, X_val_seq, y_val)
test_ds = HybridFraudDataset(X_test_img, X_test_seq, y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

print("Train final:", len(train_ds), "| fraud:", int(y_train_final.sum()))
print("Val:", len(val_ds), "| fraud:", int(y_val.sum()))
print("Test:", len(test_ds), "| fraud:", int(y_test.sum()))

In [ ]:
# ============================================================
# 16. VIT IMPLEMENTATION
# ============================================================

class ViTBranch(nn.Module):
    def __init__(self, img_size=8, patch_size=2, embed_dim=64, heads=4, depth=4):
        super().__init__()
        assert img_size % patch_size == 0
        self.patch_embed = nn.Conv2d(1, embed_dim, kernel_size=patch_size, stride=patch_size)
        num_patches = (img_size // patch_size) ** 2
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=heads,
            dim_feedforward=embed_dim*4,
            dropout=0.1,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.last_patch_tokens = None

    def forward(self, x):
        x = self.patch_embed(x).flatten(2).transpose(1,2)
        cls = self.cls_token.expand(x.size(0), -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = x + self.pos_embed[:, :x.size(1)]
        x = self.encoder(x)
        x = self.norm(x)
        self.last_patch_tokens = x
        return x[:,0]

In [ ]:
# ============================================================
# 17. SWIN TRANSFORMER — 2 STAGES, 2×2 WINDOWS, 64→128
# ============================================================

def window_partition(x, window_size):
    B, H, W, C = x.shape
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    windows = x.permute(0,1,3,2,4,5).contiguous().view(-1, window_size*window_size, C)
    return windows


def window_reverse(windows, window_size, H, W, B):
    C = windows.shape[-1]
    x = windows.view(B, H//window_size, W//window_size, window_size, window_size, C)
    x = x.permute(0,1,3,2,4,5).contiguous().view(B,H,W,C)
    return x


def build_attn_mask(H, W, ws, shift, device):
    if shift == 0:
        return None
    img_mask = torch.zeros((1,H,W,1), device=device)
    h_slices = (slice(0, -ws), slice(-ws, -shift), slice(-shift, None))
    w_slices = (slice(0, -ws), slice(-ws, -shift), slice(-shift, None))
    cnt = 0
    for h in h_slices:
        for w in w_slices:
            img_mask[:,h,w,:] = cnt
            cnt += 1
    mask_windows = window_partition(img_mask, ws).view(-1, ws*ws)
    attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
    attn_mask = attn_mask.masked_fill(attn_mask != 0, -100.0).masked_fill(attn_mask == 0, 0.0)
    return attn_mask


class WindowSelfAttention(nn.Module):
    def __init__(self, dim, num_heads, window_size=2):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.window_size = window_size
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim*3)
        self.proj = nn.Linear(dim, dim)
        num_relative = (2*window_size-1)*(2*window_size-1)
        self.relative_position_bias_table = nn.Parameter(torch.zeros(num_relative, num_heads))
        coords = torch.stack(torch.meshgrid(torch.arange(window_size), torch.arange(window_size), indexing="ij"))
        coords_flat = coords.flatten(1)
        rel = coords_flat[:,:,None] - coords_flat[:,None,:]
        rel = rel.permute(1,2,0).contiguous()
        rel[:,:,0] += window_size-1
        rel[:,:,1] += window_size-1
        rel[:,:,0] *= 2*window_size-1
        relative_index = rel.sum(-1)
        self.register_buffer("relative_position_index", relative_index)
        nn.init.trunc_normal_(self.relative_position_bias_table, std=0.02)

    def forward(self, x, mask=None):
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_,N,3,self.num_heads,self.head_dim).permute(2,0,3,1,4)
        q,k,v = qkv[0],qkv[1],qkv[2]
        q = q * self.scale
        attn = q @ k.transpose(-2,-1)
        bias = self.relative_position_bias_table[self.relative_position_index.view(-1)]
        bias = bias.view(N,N,-1).permute(2,0,1).unsqueeze(0)
        attn = attn + bias
        if mask is not None:
            nw = mask.shape[0]
            attn = attn.view(B_//nw, nw, self.num_heads, N, N) + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)
        attn = torch.softmax(attn, dim=-1)
        x = (attn @ v).transpose(1,2).reshape(B_,N,C)
        return self.proj(x)


class SwinBlock(nn.Module):
    def __init__(self, dim, heads, window_size=2, shift=False):
        super().__init__()
        self.dim = dim
        self.ws = window_size
        self.shift = window_size//2 if shift else 0
        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowSelfAttention(dim, heads, window_size)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim*4), nn.GELU(), nn.Dropout(0.1), nn.Linear(dim*4, dim), nn.Dropout(0.1)
        )

    def forward(self, x, H, W):
        B,L,C = x.shape
        shortcut = x
        x = self.norm1(x).view(B,H,W,C)
        if self.shift > 0:
            x = torch.roll(x, shifts=(-self.shift,-self.shift), dims=(1,2))
        windows = window_partition(x, self.ws)
        mask = build_attn_mask(H,W,self.ws,self.shift,x.device)
        windows = self.attn(windows, mask=mask)
        x = window_reverse(windows, self.ws, H,W,B)
        if self.shift > 0:
            x = torch.roll(x, shifts=(self.shift,self.shift), dims=(1,2))
        x = x.view(B,L,C)
        x = shortcut + x
        x = x + self.mlp(self.norm2(x))
        return x


class PatchMerging(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.norm = nn.LayerNorm(dim*4)
        self.reduction = nn.Linear(dim*4, dim*2, bias=False)
    def forward(self, x, H,W):
        B,L,C = x.shape
        x = x.view(B,H,W,C)
        x0=x[:,0::2,0::2]; x1=x[:,1::2,0::2]; x2=x[:,0::2,1::2]; x3=x[:,1::2,1::2]
        x=torch.cat([x0,x1,x2,x3],dim=-1)
        H2,W2=x.shape[1],x.shape[2]
        x=self.reduction(self.norm(x.view(B,H2*W2,C*4)))
        return x,H2,W2


class SwinBranch(nn.Module):
    def __init__(self, img_size=8, patch_size=2, window_size=2):
        super().__init__()
        self.patch_embed = nn.Conv2d(1,64,kernel_size=patch_size,stride=patch_size)
        self.stage1 = nn.ModuleList([
            SwinBlock(64,4,window_size,shift=False),
            SwinBlock(64,4,window_size,shift=True)
        ])
        self.merge = PatchMerging(64)
        self.stage2 = nn.ModuleList([
            SwinBlock(128,4,window_size,shift=False),
            SwinBlock(128,4,window_size,shift=True)
        ])
        self.norm = nn.LayerNorm(128)
        self.project = nn.Linear(128,64)

    def forward(self, image):
        x = self.patch_embed(image).permute(0,2,3,1)
        B,H,W,C = x.shape
        x=x.view(B,H*W,C)
        for block in self.stage1:
            x=block(x,H,W)
        x,H,W = self.merge(x,H,W)
        for block in self.stage2:
            x=block(x,H,W)
        x=self.norm(x).mean(1)
        return self.project(x)

In [ ]:
!pip install mamba_ssm

In [ ]:
# ============================================================
# 18. MAMBA BRANCH — OFFICIAL MAMBA IF AVAILABLE
# ============================================================

MAMBA_BACKEND = "unknown"
MambaOfficial = None
try:
    from mamba_ssm import Mamba as MambaOfficial
    MAMBA_BACKEND = "official_mamba_ssm"
except Exception as e:
    MAMBA_BACKEND = f"unavailable: {type(e).__name__}: {e}"

print("Mamba backend:", MAMBA_BACKEND)
if RUN_PROFILE == "PAPER" and REQUIRE_OFFICIAL_MAMBA and MambaOfficial is None:
    raise ImportError("Paper mode requires the official mamba_ssm backend, but import failed. Check the Colab PyTorch/CUDA compatibility and reinstall mamba-ssm.")


class FallbackSSMBlock(nn.Module):
    """Explicitly labeled approximation of a state-space recurrent branch."""
    def __init__(self, d_model=64, state_dim=16):
        super().__init__()
        self.in_proj = nn.Linear(d_model, d_model)
        self.A_log = nn.Parameter(torch.zeros(d_model))
        self.B = nn.Parameter(torch.randn(d_model)*0.02)
        self.C = nn.Parameter(torch.randn(d_model)*0.02)
        self.D = nn.Parameter(torch.ones(d_model))
        self.norm = nn.LayerNorm(d_model)
    def forward(self, u):
        B,L,D=u.shape
        A=-F.softplus(self.A_log)
        state=torch.zeros(B,D,device=u.device,dtype=u.dtype)
        outs=[]
        u=self.in_proj(u)
        for t in range(L):
            ut=u[:,t]
            state=torch.exp(A)*state+self.B*ut
            yt=self.C*state+self.D*ut
            outs.append(yt.unsqueeze(1))
        return self.norm(torch.cat(outs,dim=1))


class MambaBranch(nn.Module):
    def __init__(self, input_dim=64, d_model=64, state_dim=16, layers=2, require_official=True):
        super().__init__()
        self.input_proj=nn.Linear(input_dim,d_model)
        self.require_official=require_official
        if MambaOfficial is not None:
            self.blocks=nn.ModuleList([
                MambaOfficial(d_model=d_model,d_state=state_dim,d_conv=4,expand=2)
                for _ in range(layers)
            ])
        else:
            if require_official:
                raise ImportError("Official mamba_ssm could not be imported. Install it or set REQUIRE_OFFICIAL_MAMBA=False explicitly.")
            self.blocks=nn.ModuleList([
                FallbackSSMBlock(d_model=d_model,state_dim=state_dim)
                for _ in range(layers)
            ])
        self.norm=nn.LayerNorm(d_model)

    def forward(self, seq):
        x=self.input_proj(seq)
        for block in self.blocks:
            x=x+block(x)
        x=self.norm(x)
        return x[:,-1]

In [ ]:
# ============================================================
# 19. DFFM + CLASSIFIER
# ============================================================

class DFFM(nn.Module):
    def __init__(self, feature_dim=64):
        super().__init__()
        self.gate = nn.Linear(feature_dim*3,3)
    def forward(self, f_vit, f_swin, f_mamba):
        z=torch.cat([f_vit,f_swin,f_mamba],dim=1)
        weights=torch.softmax(self.gate(z),dim=1)
        fused=(weights[:,0:1]*f_vit + weights[:,1:2]*f_swin + weights[:,2:3]*f_mamba)
        return fused,weights


def make_classifier(input_dim, dropout=0.3):
    # Manuscript reference: FC(256) -> FC(128) -> FC(64), dropout 0.3.
    return nn.Sequential(
        nn.Linear(input_dim,256), nn.GELU(), nn.Dropout(dropout),
        nn.Linear(256,128), nn.GELU(), nn.Dropout(dropout),
        nn.Linear(128,64), nn.GELU(), nn.Dropout(dropout),
        nn.Linear(64,2)
    )

In [ ]:
# ============================================================
# 20. UNIFIED MODEL FOR ALL 8 ABLATIONS
# ============================================================

class DFFMVSMModel(nn.Module):
    def __init__(self, use_vit=True, use_swin=True, use_mamba=True, use_dffm=False):
        super().__init__()
        self.use_vit=use_vit
        self.use_swin=use_swin
        self.use_mamba=use_mamba
        self.use_dffm=use_dffm

        if use_vit:
            self.vit=ViTBranch(8,2,64,4,4)
        if use_swin:
            self.swin=SwinBranch(8,2,2)
        if use_mamba:
            self.mamba=MambaBranch(64,64,16,2,REQUIRE_OFFICIAL_MAMBA)

        branch_count=sum([use_vit,use_swin,use_mamba])
        if branch_count == 0:
            raise ValueError("At least one branch is required.")

        if use_dffm:
            if branch_count != 3:
                raise ValueError("DFFM requires all three branches.")
            self.dffm=DFFM(64)
            head_input=64
        else:
            head_input=64*branch_count

        self.classifier=make_classifier(head_input,dropout=0.3)

    def forward(self,image,sequence):
        fv=None; fs=None; fm=None
        feats=[]
        if self.use_vit:
            fv=self.vit(image); feats.append(fv)
        if self.use_swin:
            fs=self.swin(image); feats.append(fs)
        if self.use_mamba:
            fm=self.mamba(sequence); feats.append(fm)

        weights=None
        if self.use_dffm:
            fused,weights=self.dffm(fv,fs,fm)
        else:
            fused=torch.cat(feats,dim=1)
        logits=self.classifier(fused)
        return logits,weights

## 20B. Architecture smoke test before expensive training

This cell creates one random batch and verifies that all eight configurations can execute a forward pass. If the official Mamba backend is unavailable in the selected profile, the notebook stops before spending time on training rather than silently switching architectures.

In [ ]:
# ============================================================
# 20B. FORWARD-PASS SMOKE TEST FOR ALL 8 CONFIGURATIONS
# ============================================================

if DEVICE.type != "cuda" and REQUIRE_OFFICIAL_MAMBA:
    print("WARNING: Official Mamba is requested but CUDA is unavailable. "
          "For the paper configuration, use a Colab GPU runtime.")

smoke_batch = 2
smoke_image = torch.randn(smoke_batch, 1, 8, 8, device=DEVICE)
smoke_sequence = torch.randn(smoke_batch, SEQUENCE_LENGTH, 64, device=DEVICE)

for name, cfg in {
    "ViT-only": dict(use_vit=True,use_swin=False,use_mamba=False,use_dffm=False),
    "Swin-only": dict(use_vit=False,use_swin=True,use_mamba=False,use_dffm=False),
    "Mamba-only": dict(use_vit=False,use_swin=False,use_mamba=True,use_dffm=False),
    "ViT+Swin": dict(use_vit=True,use_swin=True,use_mamba=False,use_dffm=False),
    "ViT+Mamba": dict(use_vit=True,use_swin=False,use_mamba=True,use_dffm=False),
    "Swin+Mamba": dict(use_vit=False,use_swin=True,use_mamba=True,use_dffm=False),
    "ViT+Swin+Mamba (concat.)": dict(use_vit=True,use_swin=True,use_mamba=True,use_dffm=False),
    "DFFM-VSM (proposed)": dict(use_vit=True,use_swin=True,use_mamba=True,use_dffm=True),
}.items():
    m = DFFMVSMModel(**cfg).to(DEVICE)
    with torch.no_grad():
        logits, weights = m(smoke_image, smoke_sequence)
    assert logits.shape == (smoke_batch, 2)
    if weights is not None:
        assert weights.shape == (smoke_batch, 3)
        assert torch.allclose(weights.sum(dim=1), torch.ones(smoke_batch, device=DEVICE), atol=1e-5)
    del m
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    print(f"PASS: {name}")

print("All eight configurations passed the forward-pass smoke test.")


In [ ]:
# ============================================================
# 21. CNN + LSTM BASELINES FROM THE MANUSCRIPT'S S1 SETUP
# ============================================================

class CNNBaseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.ReLU(), nn.BatchNorm2d(32),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.BatchNorm2d(64),
            nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )
        self.head=make_classifier(128,0.3)
    def forward(self,image,sequence):
        x=self.net(image).flatten(1)
        return self.head(x),None

class LSTMBaseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.input=nn.Linear(64,64)
        self.lstm=nn.LSTM(64,64,num_layers=2,batch_first=True,dropout=0.2)
        self.norm=nn.LayerNorm(64)
        self.head=make_classifier(64,0.3)
    def forward(self,image,sequence):
        x=self.input(sequence)
        x,_=self.lstm(x)
        x=self.norm(x[:,-1])
        return self.head(x),None

In [ ]:
# ============================================================
# 22. TRAINING + EVALUATION UTILITIES
# ============================================================

def make_optimizer(model):
    return torch.optim.Adam(
        model.parameters(),
        lr=LR,
        betas=(0.9,0.999),
        weight_decay=WEIGHT_DECAY
    )


def get_amp_tools():
    enabled = USE_MIXED_PRECISION and torch.cuda.is_available()
    scaler = torch.cuda.amp.GradScaler(enabled=enabled)
    return enabled, scaler


def train_model(model, train_loader, val_loader, name, epochs=EPOCHS, patience=PATIENCE):
    model=model.to(DEVICE)
    criterion=nn.CrossEntropyLoss()
    optimizer=make_optimizer(model)
    amp_enabled, scaler=get_amp_tools()
    history={"train_loss":[],"val_loss":[]}
    best_val=float("inf")
    best_state=None
    patience_count=0

    ckpt_path=OUTPUT_DIR/f"{name.replace('/','_').replace('+','_')}.pth"

    for epoch in range(epochs):
        model.train()
        running=0.0
        count=0
        for image,seq,y in tqdm(train_loader,desc=f"{name} train {epoch+1}/{epochs}",leave=False):
            image=image.to(DEVICE,non_blocking=True)
            seq=seq.to(DEVICE,non_blocking=True)
            y=y.to(DEVICE,non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=amp_enabled):
                logits,_=model(image,seq)
                loss=criterion(logits,y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            running += loss.item()*len(y)
            count += len(y)
        train_loss=running/max(1,count)

        model.eval()
        vrunning=0.0; vcount=0
        with torch.no_grad():
            for image,seq,y in val_loader:
                image=image.to(DEVICE,non_blocking=True)
                seq=seq.to(DEVICE,non_blocking=True)
                y=y.to(DEVICE,non_blocking=True)
                logits,_=model(image,seq)
                loss=criterion(logits,y)
                vrunning += loss.item()*len(y)
                vcount += len(y)
        val_loss=vrunning/max(1,vcount)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        print(f"{name} | epoch {epoch+1:03d} | train={train_loss:.6f} | val={val_loss:.6f}")

        if val_loss < best_val:
            best_val=val_loss
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            patience_count=0
        else:
            patience_count += 1
            if patience_count >= patience:
                print("Early stopping")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        torch.save(best_state,ckpt_path)

    with open(OUTPUT_DIR/f"{name.replace('/','_').replace('+','_')}_history.json","w") as f:
        json.dump(history,f,indent=2)
    return model,history


def evaluate_model(model, loader, threshold=THRESHOLD):
    model.eval()
    y_true=[]; y_pred=[]; y_prob=[]; weight_list=[]
    with torch.no_grad():
        for image,seq,y in tqdm(loader,desc="evaluate",leave=False):
            image=image.to(DEVICE,non_blocking=True)
            seq=seq.to(DEVICE,non_blocking=True)
            logits,weights=model(image,seq)
            p=torch.softmax(logits,dim=1)[:,1]
            y_true.extend(y.numpy())
            y_prob.extend(p.cpu().numpy())
            y_pred.extend((p>=threshold).long().cpu().numpy())
            if weights is not None:
                weight_list.append(weights.cpu().numpy())

    y_true=np.asarray(y_true); y_pred=np.asarray(y_pred); y_prob=np.asarray(y_prob)
    cm=confusion_matrix(y_true,y_pred,labels=[0,1])
    tn,fp,fn,tp=cm.ravel()
    specificity=tn/(tn+fp) if tn+fp else 0.0
    fpr=fp/(fp+tn) if fp+tn else 0.0
    metrics={
        "Accuracy":accuracy_score(y_true,y_pred),
        "Precision":precision_score(y_true,y_pred,zero_division=0),
        "Recall":recall_score(y_true,y_pred,zero_division=0),
        "F1":f1_score(y_true,y_pred,zero_division=0),
        "ROC_AUC":roc_auc_score(y_true,y_prob),
        "PR_AUC":average_precision_score(y_true,y_prob),
        "MCC":matthews_corrcoef(y_true,y_pred),
        "Balanced_Accuracy":balanced_accuracy_score(y_true,y_pred),
        "Specificity":specificity,
        "FPR":fpr,
        "TP":int(tp),"TN":int(tn),"FP":int(fp),"FN":int(fn),
    }
    weights=np.concatenate(weight_list,axis=0) if weight_list else None
    return metrics,cm,y_true,y_pred,y_prob,weights

In [ ]:
# ============================================================
# 23. RUN 8 ABLATION EXPERIMENTS
# ============================================================

EXPERIMENTS = {
    "ViT-only": dict(use_vit=True,use_swin=False,use_mamba=False,use_dffm=False),
    "Swin-only": dict(use_vit=False,use_swin=True,use_mamba=False,use_dffm=False),
    "Mamba-only": dict(use_vit=False,use_swin=False,use_mamba=True,use_dffm=False),
    "ViT+Swin": dict(use_vit=True,use_swin=True,use_mamba=False,use_dffm=False),
    "ViT+Mamba": dict(use_vit=True,use_swin=False,use_mamba=True,use_dffm=False),
    "Swin+Mamba": dict(use_vit=False,use_swin=True,use_mamba=True,use_dffm=False),
    "ViT+Swin+Mamba (concat.)": dict(use_vit=True,use_swin=True,use_mamba=True,use_dffm=False),
    "DFFM-VSM (proposed)": dict(use_vit=True,use_swin=True,use_mamba=True,use_dffm=True),
}

actual_ablation_results=[]
actual_ablation_curves={}
actual_ablation_histories={}
actual_ablation_cms={}
actual_dffm_weights=None

if RUN_ABLATIONS and RUN_PROFILE != "AUDIT_ONLY":
    for name, cfg in EXPERIMENTS.items():
        print("\n"+"="*90)
        print("RUNNING:", name)
        print("="*90)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        model=DFFMVSMModel(**cfg)
        params=sum(p.numel() for p in model.parameters() if p.requires_grad)
        model,history=train_model(model,train_loader,val_loader,name)
        metrics,cm,yt,yp,yprob,weights=evaluate_model(model,test_loader)
        metrics["Model"]=name
        metrics["Parameters"]=params
        actual_ablation_results.append(metrics)
        actual_ablation_histories[name]=history
        actual_ablation_cms[name]=cm
        actual_ablation_curves[name]=(yt,yprob)
        if weights is not None:
            actual_dffm_weights=weights
        print(metrics)

    actual_ablation_df=pd.DataFrame(actual_ablation_results)
    ordered_cols=["Model","Accuracy","PR_AUC","MCC","Balanced_Accuracy","FPR","Recall","Specificity","Precision","F1","ROC_AUC","TP","TN","FP","FN","Parameters"]
    actual_ablation_df=actual_ablation_df[ordered_cols]
    actual_ablation_df.to_csv(OUTPUT_DIR/"ablation_results_actual.csv",index=False)
    print(actual_ablation_df.to_string(index=False,float_format=lambda x:f"{x:.6f}"))
else:
    actual_ablation_df=pd.DataFrame()
    print("Ablation training disabled.")

In [ ]:
# ============================================================
# 24. PLOT ACTUAL ABLATION RESULTS
# ============================================================

if not actual_ablation_df.empty:
    # Accuracy + PR-AUC
    p=actual_ablation_df.copy()
    x=np.arange(len(p)); width=0.36
    plt.figure(figsize=(13,6))
    plt.bar(x-width/2,p["Accuracy"],width,label="Accuracy")
    plt.bar(x+width/2,p["PR_AUC"],width,label="PR-AUC")
    plt.xticks(x,p["Model"],rotation=35,ha="right")
    plt.ylabel("Score")
    plt.title("Actual Ablation Study — Accuracy and PR-AUC")
    plt.legend(); plt.grid(axis="y",alpha=0.25); plt.tight_layout()
    plt.savefig(OUTPUT_DIR/"ablation_accuracy_pr_auc_actual.png",dpi=300,bbox_inches="tight")
    plt.show()

    # Fraud-focused metrics
    plt.figure(figsize=(13,6))
    for m in ["MCC","Balanced_Accuracy","Recall","Specificity","FPR"]:
        plt.plot(x,p[m],marker="o",label=m)
    plt.xticks(x,p["Model"],rotation=35,ha="right")
    plt.ylabel("Score")
    plt.title("Actual Fraud-Focused Metrics Across Ablations")
    plt.legend(); plt.grid(alpha=0.25); plt.tight_layout()
    plt.savefig(OUTPUT_DIR/"ablation_fraud_metrics_actual.png",dpi=300,bbox_inches="tight")
    plt.show()

In [ ]:
# ============================================================
# 25. PLOT MANUSCRIPT-REPORTED ABLATION VALUES FOR DIRECT REFERENCE
# ============================================================

p=MANUSCRIPT_ABLATION.copy()
x=np.arange(len(p)); width=0.36
plt.figure(figsize=(13,6))
plt.bar(x-width/2,p["Accuracy_pct"],width,label="Accuracy (%)")
plt.bar(x+width/2,p["PR_AUC"]*100,width,label="PR-AUC × 100")
plt.xticks(x,p["Model"],rotation=35,ha="right")
plt.ylabel("Reported value")
plt.title("Manuscript-Reported Ablation Values — Reference Only")
plt.legend(); plt.grid(axis="y",alpha=0.25); plt.tight_layout()
plt.savefig(OUTPUT_DIR/"manuscript_ablation_reference.png",dpi=300,bbox_inches="tight")
plt.show()

MANUSCRIPT_ABLATION.to_csv(OUTPUT_DIR/"manuscript_reported_ablation_reference.csv",index=False)
MANUSCRIPT_BASELINE.to_csv(OUTPUT_DIR/"manuscript_reported_baseline_reference.csv",index=False)
MANUSCRIPT_CV.to_csv(OUTPUT_DIR/"manuscript_reported_cv_reference.csv",index=False)
MANUSCRIPT_CROSSDATA.to_csv(OUTPUT_DIR/"manuscript_reported_crossdataset_reference.csv",index=False)
MANUSCRIPT_SHAP.to_csv(OUTPUT_DIR/"manuscript_reported_shap_reference.csv",index=False)

In [ ]:
# ============================================================
# 26. ACTUAL CONFUSION MATRICES + PR/ROC CURVES
# ============================================================

if actual_ablation_cms:
    for name,cm in actual_ablation_cms.items():
        plt.figure(figsize=(5,4))
        sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",xticklabels=["Legitimate","Fraud"],yticklabels=["Legitimate","Fraud"])
        plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(f"Confusion Matrix — {name}")
        plt.tight_layout()
        fn=name.replace(" ","_").replace("+","_").replace("/","_").replace(".","").lower()
        plt.savefig(OUTPUT_DIR/f"confusion_matrix_{fn}.png",dpi=300,bbox_inches="tight")
        plt.show(); plt.close()

    plt.figure(figsize=(9,7))
    for name,(yt,yp) in actual_ablation_curves.items():
        precision,recall,_=precision_recall_curve(yt,yp)
        pr=average_precision_score(yt,yp)
        plt.plot(recall,precision,label=f"{name} (PR-AUC={pr:.4f})")
    plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("Precision–Recall Curves — Actual Runs")
    plt.legend(fontsize=8); plt.grid(alpha=0.25); plt.tight_layout()
    plt.savefig(OUTPUT_DIR/"precision_recall_curves_actual.png",dpi=300,bbox_inches="tight"); plt.show(); plt.close()

    plt.figure(figsize=(9,7))
    for name,(yt,yp) in actual_ablation_curves.items():
        fpr,tpr,_=roc_curve(yt,yp)
        auc=roc_auc_score(yt,yp)
        plt.plot(fpr,tpr,label=f"{name} (AUC={auc:.4f})")
    plt.plot([0,1],[0,1],"--",linewidth=1)
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate"); plt.title("ROC Curves — Actual Runs")
    plt.legend(fontsize=8); plt.grid(alpha=0.25); plt.tight_layout()
    plt.savefig(OUTPUT_DIR/"roc_curves_actual.png",dpi=300,bbox_inches="tight"); plt.show(); plt.close()

In [ ]:
# ============================================================
# 27. LOSS CURVES
# ============================================================

for name,history in actual_ablation_histories.items():
    plt.figure(figsize=(7,5))
    plt.plot(history["train_loss"],marker="o",label="Training")
    plt.plot(history["val_loss"],marker="s",label="Validation")
    plt.xlabel("Epoch"); plt.ylabel("Cross-Entropy Loss"); plt.title(f"Training/Validation Loss — {name}")
    plt.legend(); plt.grid(alpha=0.25); plt.tight_layout()
    fn=name.replace(" ","_").replace("+","_").replace("/","_").replace(".","").lower()
    plt.savefig(OUTPUT_DIR/f"loss_curve_{fn}.png",dpi=300,bbox_inches="tight"); plt.show(); plt.close()

In [ ]:
# ============================================================
# 28. DFFM ADAPTIVE WEIGHT ANALYSIS
# ============================================================

if actual_dffm_weights is not None:
    mean_w=actual_dffm_weights.mean(axis=0)
    std_w=actual_dffm_weights.std(axis=0)
    weights_df=pd.DataFrame({"Branch":["ViT","Swin","Mamba"],"MeanWeight":mean_w,"StdWeight":std_w})
    weights_df.to_csv(OUTPUT_DIR/"dffm_adaptive_weights_actual.csv",index=False)

    plt.figure(figsize=(7,5))
    plt.bar(weights_df["Branch"],weights_df["MeanWeight"],yerr=weights_df["StdWeight"],capsize=5)
    plt.ylabel("Adaptive fusion weight")
    plt.title("DFFM Transaction-Adaptive Fusion Weights — Actual Run")
    plt.grid(axis="y",alpha=0.25); plt.tight_layout()
    plt.savefig(OUTPUT_DIR/"dffm_adaptive_weights_actual.png",dpi=300,bbox_inches="tight"); plt.show()
else:
    print("No actual DFFM weights available because the proposed model was not trained in this run.")

In [ ]:
# ============================================================
# 29. CNN/LSTM BASELINES
# ============================================================

baseline_actual=[]
if RUN_BASELINES and RUN_PROFILE != "AUDIT_ONLY":
    for name,model in [
        ("CNN",CNNBaseline()),
        ("LSTM",LSTMBaseline()),
    ]:
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        model,history=train_model(model,train_loader,val_loader,name)
        metrics,cm,yt,yp,yprob,weights=evaluate_model(model,test_loader)
        metrics["Model"]=name
        baseline_actual.append(metrics)
        print(name,metrics)
    baseline_actual_df=pd.DataFrame(baseline_actual)
    baseline_actual_df.to_csv(OUTPUT_DIR/"baseline_results_actual.csv",index=False)
    print(baseline_actual_df.to_string(index=False,float_format=lambda x:f"{x:.6f}"))
else:
    baseline_actual_df=pd.DataFrame()

## 30. 5-fold chronological cross-validation

The manuscript specifies five **contiguous chronological folds** and explicitly rejects training on transactions occurring after a fold's test block. The exact fold boundaries are not supplied, so this notebook uses `TimeSeriesSplit(n_splits=5)` over the available training/validation pool and records that choice in the output. This is a reproducible formalization of the manuscript's stated temporal constraint, not a claim that it is the hidden original fold partition.

In [ ]:
# ============================================================
# 30. 5-FOLD CHRONOLOGICAL CV (OPTIONAL, EXPENSIVE)
# ============================================================

cv_results=[]

if RUN_CV and RUN_PROFILE != "AUDIT_ONLY":
    # To keep fold datasets chronological and leakage-safe, use the original train+val+test order
    # but only evaluate each test block after the preceding blocks. For a practical notebook run,
    # use a configurable sample of the training/validation pool.
    X_cv = np.concatenate([X_train64, X_val64, X_test64],axis=0)
    y_cv = np.concatenate([y_train, y_val, y_test],axis=0)
    img_cv = X_cv.reshape(-1,1,8,8)

    # Build a simple global past-only sequence for CV evaluation if account-specific arrays are unavailable.
    seq_cv = np.zeros((len(X_cv),SEQUENCE_LENGTH,64),dtype=np.float32)
    for i in range(len(X_cv)):
        start=max(0,i-SEQUENCE_LENGTH)
        hist=X_cv[start:i]
        if len(hist)>0:
            seq_cv[i,-len(hist):]=hist

    tscv=TimeSeriesSplit(n_splits=5)
    for fold,(tr_idx,te_idx) in enumerate(tscv.split(img_cv),1):
        # Keep training before the fold test block; reserve the last 15% of the training portion for validation.
        if len(tr_idx)<100:
            continue
        cut=max(1,int(len(tr_idx)*0.85))
        tr=tr_idx[:cut]
        va=tr_idx[cut:]
        if len(va)==0:
            va=tr_idx[-1:]
            tr=tr_idx[:-1]

        tr_ds=HybridFraudDataset(img_cv[tr],seq_cv[tr],y_cv[tr])
        va_ds=HybridFraudDataset(img_cv[va],seq_cv[va],y_cv[va])
        te_ds=HybridFraudDataset(img_cv[te_idx],seq_cv[te_idx],y_cv[te_idx])
        tr_ld=DataLoader(tr_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=torch.cuda.is_available())
        va_ld=DataLoader(va_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=torch.cuda.is_available())
        te_ld=DataLoader(te_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=torch.cuda.is_available())

        model=DFFMVSMModel(True,True,True,True)
        model,_=train_model(model,tr_ld,va_ld,f"CV_Fold_{fold}",epochs=min(EPOCHS,30),patience=min(PATIENCE,5))
        metrics,_,_,_,_,_=evaluate_model(model,te_ld)
        cv_results.append({"Fold":fold,**metrics})
        print("Fold",fold,metrics)
        del tr_ds,va_ds,te_ds,tr_ld,va_ld,te_ld,model
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    cv_df=pd.DataFrame(cv_results)
    if not cv_df.empty:
        cv_df.to_csv(OUTPUT_DIR/"cross_validation_results_actual.csv",index=False)
        print(cv_df.to_string(index=False,float_format=lambda x:f"{x:.6f}"))
        print("Mean ± Std Accuracy:",cv_df.Accuracy.mean(),"±",cv_df.Accuracy.std(ddof=1) if len(cv_df)>1 else 0)
else:
    cv_df=pd.DataFrame()
    print("5-fold CV disabled for this run profile.")

In [ ]:
# ============================================================
# 31. TRANSFORMER-STYLE GRAD-CAM FOR THE VIT PATCH TOKENS
# ============================================================

def vit_gradcam(model, image, target_class=1):
    model.eval()
    image=image.to(DEVICE)
    model.zero_grad(set_to_none=True)
    logits,_=model(image, torch.zeros((image.size(0),SEQUENCE_LENGTH,64),device=DEVICE))
    score=logits[:,target_class].sum()
    model.vit.last_patch_tokens.retain_grad()
    score.backward()
    tokens=model.vit.last_patch_tokens  # [B,17,64] includes class token
    grads=tokens.grad[:,1:]
    acts=tokens[:,1:]
    weights=grads.mean(dim=1,keepdim=True)
    cam=(weights*acts).sum(dim=-1)
    cam=F.relu(cam)
    cam=cam/(cam.amax(dim=1,keepdim=True)+1e-8)
    cam=cam.reshape(-1,4,4)
    cam=F.interpolate(cam.unsqueeze(1),size=(8,8),mode="bilinear",align_corners=False).squeeze(1)
    return cam.detach().cpu().numpy()


if RUN_XAI and not actual_ablation_df.empty and "DFFM-VSM (proposed)" in actual_ablation_df["Model"].values:
    xai_model=DFFMVSMModel(True,True,True,True).to(DEVICE)
    ckpt=OUTPUT_DIR/"DFFM-VSM_(proposed).pth"
    if ckpt.exists():
        xai_model.load_state_dict(torch.load(ckpt,map_location=DEVICE))
        sample_n=min(4,len(test_ds))
        sample_images=torch.tensor(X_test_img[:sample_n],dtype=torch.float32)
        cams=vit_gradcam(xai_model,sample_images,target_class=1)
        for i,cam in enumerate(cams):
            plt.figure(figsize=(5,5))
            plt.imshow(X_test_img[i,0],cmap="gray")
            plt.imshow(cam,cmap="jet",alpha=0.45)
            plt.axis("off")
            plt.title(f"Grad-CAM — Test Transaction {i}")
            plt.tight_layout()
            plt.savefig(OUTPUT_DIR/f"gradcam_test_{i}.png",dpi=300,bbox_inches="tight")
            plt.show(); plt.close()
        np.save(OUTPUT_DIR/"gradcam_maps.npy",cams)
    del xai_model

In [ ]:
# ============================================================
# 32. SHAP GLOBAL FEATURE IMPORTANCE (OPTIONAL / EXPENSIVE)
# ============================================================

if RUN_XAI and not actual_ablation_df.empty and "DFFM-VSM (proposed)" in actual_ablation_df["Model"].values:
    try:
        import shap
        shap_model=DFFMVSMModel(True,True,True,True).to(DEVICE)
        ckpt=OUTPUT_DIR/"DFFM-VSM_(proposed).pth"
        if ckpt.exists():
            shap_model.load_state_dict(torch.load(ckpt,map_location=DEVICE))

            bg_n=min(40,len(X_train64))
            explain_n=min(20,len(X_test64))
            bg=X_train64[np.random.default_rng(SEED).choice(len(X_train64),bg_n,replace=False)]
            explain=X_test64[:explain_n]

            zero_seq=np.zeros((len(bg),SEQUENCE_LENGTH,64),dtype=np.float32)
            zero_seq_ex=np.zeros((len(explain),SEQUENCE_LENGTH,64),dtype=np.float32)

            def predict_prob(z):
                z=np.asarray(z,dtype=np.float32)
                imgs=z.reshape(-1,1,8,8)
                seq=np.zeros((len(z),SEQUENCE_LENGTH,64),dtype=np.float32)
                preds=[]
                with torch.no_grad():
                    for s in range(0,len(z),32):
                        im=torch.tensor(imgs[s:s+32],dtype=torch.float32,device=DEVICE)
                        sq=torch.tensor(seq[s:s+32],dtype=torch.float32,device=DEVICE)
                        logits,_=shap_model(im,sq)
                        preds.append(torch.softmax(logits,dim=1)[:,1].cpu().numpy())
                return np.concatenate(preds)

            explainer=shap.Explainer(predict_prob,bg,algorithm="permutation")
            sv=explainer(explain,max_evals=min(2*64+1,129))
            mean_abs=np.abs(sv.values).mean(axis=0)
            shap_df=pd.DataFrame({
                "Feature":selected,
                "MeanAbsSHAP":mean_abs,
            }).sort_values("MeanAbsSHAP",ascending=False)
            shap_df["RelativeImportance"]=shap_df["MeanAbsSHAP"]/(shap_df["MeanAbsSHAP"].max()+1e-12)
            shap_df.to_csv(OUTPUT_DIR/"shap_feature_importance_actual.csv",index=False)

            plt.figure(figsize=(10,6))
            top=shap_df.head(15).iloc[::-1]
            plt.barh(top["Feature"],top["MeanAbsSHAP"])
            plt.xlabel("Mean absolute SHAP value")
            plt.title("Global SHAP Feature Importance — Actual Run")
            plt.tight_layout()
            plt.savefig(OUTPUT_DIR/"shap_feature_importance_actual.png",dpi=300,bbox_inches="tight")
            plt.show()
        else:
            print("Proposed model checkpoint not found.")
    except Exception as e:
        print("SHAP step skipped due to:",repr(e))

In [ ]:
# ============================================================
# 33. MANUSCRIPT-REPORTED SHAP REFERENCE FIGURE
# ============================================================

m=MANUSCRIPT_SHAP.sort_values("SHAP_Importance")
plt.figure(figsize=(9,5))
plt.barh(m["Feature"],m["SHAP_Importance"])
plt.xlabel("Reported normalized importance")
plt.title("Manuscript-Reported SHAP Importance — Reference Only")
plt.tight_layout()
plt.savefig(OUTPUT_DIR/"manuscript_shap_reference.png",dpi=300,bbox_inches="tight")
plt.show()

## 34. Unseen-fraud stress test

The manuscript says that unseen-fraud cases are kept in the test set, but it does not provide the exact hidden rule. On the current public schema, the most defensible reproducible proxy is **fraudulent test transactions whose sender (`nameOrig`) never appeared in the training set**. This measures fraud detection on unseen sender entities, not a perfect reconstruction of the manuscript's hidden fraud-pattern taxonomy.

In [ ]:
# ============================================================
# 34. UNSEEN-ENTITY FRAUD TEST
# ============================================================

train_entities=set(raw_df.iloc[train_idx][SEQUENCE_GROUP_PRIMARY].astype(str))
test_meta=raw_df.iloc[test_idx].copy()

unseen_mask=test_meta[SEQUENCE_GROUP_PRIMARY].astype(str).map(lambda s:s not in train_entities).to_numpy()
unseen_fraud_mask=unseen_mask & (y_test==1)
seen_fraud_mask=(~unseen_mask) & (y_test==1)

unseen_audit={
    "test_rows":int(len(test_meta)),
    "unseen_sender_rows":int(unseen_mask.sum()),
    "unseen_sender_fraud_rows":int(unseen_fraud_mask.sum()),
    "seen_sender_fraud_rows":int(seen_fraud_mask.sum()),
}

with open(OUTPUT_DIR/"unseen_fraud_audit.json","w") as f:
    json.dump(unseen_audit,f,indent=2)

print(json.dumps(unseen_audit,indent=2))

In [ ]:
# ============================================================
# 35. SYNTHETIC STRESS TEST (EXPLICITLY AN APPROXIMATION)
# ============================================================

if RUN_STRESS_TEST and not actual_ablation_df.empty and "DFFM-VSM (proposed)" in actual_ablation_df["Model"].values:
    # Construct a controlled distribution shift by perturbing amount/balance-related pixels only.
    rng=np.random.default_rng(SEED)
    stress_images=X_test_img[:min(20_000,len(X_test_img))].copy()
    stress_seq=X_test_seq[:len(stress_images)].copy()
    stress_y=y_test[:len(stress_images)].copy()

    amount_positions=[i for i,n in enumerate(selected) if "amount" in n.lower() or "balance" in n.lower()]
    if amount_positions:
        noise=rng.normal(0,0.15,stress_images.reshape(len(stress_images),64)[:,amount_positions].shape).astype(np.float32)
        stress_images_flat=stress_images.reshape(len(stress_images),64)
        stress_images_flat[:,amount_positions]+=noise
        stress_images=stress_images_flat.reshape(-1,1,8,8)

    stress_ds=HybridFraudDataset(stress_images,stress_seq,stress_y)
    stress_loader=DataLoader(stress_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS)
    stress_model=DFFMVSMModel(True,True,True,True).to(DEVICE)
    ckpt=OUTPUT_DIR/"DFFM-VSM_(proposed).pth"
    if ckpt.exists():
        stress_model.load_state_dict(torch.load(ckpt,map_location=DEVICE))
        stress_metrics,_,_,_,_,_=evaluate_model(stress_model,stress_loader)
        pd.DataFrame([stress_metrics]).to_csv(OUTPUT_DIR/"synthetic_stress_test_actual.csv",index=False)
        print(stress_metrics)
    else:
        print("Proposed checkpoint not found.")

## 35B. Manuscript-result provenance rule

This notebook intentionally keeps two result streams separate:

1. **Actual run results** — generated by the code from the public data and saved as `*_actual.csv`.
2. **Manuscript-reported results** — transcribed from the uploaded PDF and saved as `manuscript_reported_*.csv` / `manuscript_*_reference.png`.

The notebook never substitutes the manuscript numbers for the actual run. If the actual results differ, the difference is evidence that at least one unpublished implementation detail, data-preprocessing step, upstream feature-construction step, random state, software version, or resource-dependent training choice differs from the original run.

## 36. External public benchmark adapters

The manuscript's cross-dataset table names IEEE-CIS, PaySim, the European Credit Card dataset, and a synthetic stress-test set. The exact preprocessing and transfer protocol were not supplied. This notebook therefore includes loaders/adapters so that the same 64-feature → 8×8 → ViT/Swin/Mamba → DFFM pipeline can be rerun on those datasets, but **does not invent their reported numbers**.

- PaySim: place `PS_20174392719_1491204439457_log.csv` in `/content/external/paysim/` or change the path.
- European Credit Card: place `creditcard.csv` in `/content/external/european/` or change the path.
- IEEE-CIS: place `train_transaction.csv` and `train_identity.csv` in `/content/external/ieee_cis/` or change the paths.

The notebook uses chronological ordering where a time field is available. External runs are disabled by default in `COLAB` profile because downloading/processing them can be substantial.

In [ ]:
# ============================================================
# 36. EXTERNAL DATASET LOADERS
# ============================================================

EXTERNAL_ROOT=Path("/content/external")
EXTERNAL_ROOT.mkdir(exist_ok=True)


def load_paysim(path):
    df=pd.read_csv(path)
    if "isFraud" not in df.columns:
        raise ValueError("PaySim target isFraud not found")
    return df


def load_european(path):
    df=pd.read_csv(path)
    if "Class" not in df.columns:
        raise ValueError("European dataset target Class not found")
    df=df.rename(columns={"Class":"isFraud","Time":"step"})
    if "amount" not in df.columns and "Amount" in df.columns:
        df=df.rename(columns={"Amount":"amount"})
    return df


def load_ieee_cis(transaction_path, identity_path=None):
    trans=pd.read_csv(transaction_path)
    if identity_path and Path(identity_path).exists():
        ident=pd.read_csv(identity_path)
        if "TransactionID" in trans.columns and "TransactionID" in ident.columns:
            trans=trans.merge(ident,on="TransactionID",how="left")
    if "isFraud" not in trans.columns:
        raise ValueError("IEEE-CIS target isFraud not found")
    if "TransactionDT" in trans.columns:
        trans=trans.sort_values("TransactionDT",kind="stable").reset_index(drop=True)
        trans["step"]=trans["TransactionDT"]
    return trans


EXTERNAL_PATHS={
    "PaySim":"/content/external/paysim/PS_20174392719_1491204439457_log.csv",
    "European Credit Card":"/content/external/european/creditcard.csv",
    "IEEE-CIS transaction":"/content/external/ieee_cis/train_transaction.csv",
    "IEEE-CIS identity":"/content/external/ieee_cis/train_identity.csv",
}

print(json.dumps(EXTERNAL_PATHS,indent=2))

In [ ]:
# ============================================================
# 37. EXTERNAL DATASET RUNNER (OPTIONAL)
# ============================================================

def run_external_dataset(name, df):
    """Run the same architecture on an external dataset after standardization.
    This is deliberately conservative and reports actual results only when executed.
    """
    d=df.copy()
    d=d.reset_index(drop=True)
    if "isFraud" not in d.columns:
        raise ValueError(f"{name}: target isFraud missing")
    if "step" not in d.columns:
        d["step"]=np.arange(len(d))
    if "type" not in d.columns:
        d["type"]="UNKNOWN"
    if "nameOrig" not in d.columns:
        d["nameOrig"]="GLOBAL_SEQUENCE"
    if "nameDest" not in d.columns:
        d["nameDest"]="GLOBAL_DEST"
    d=clean_cifer(d)
    cand,groups=build_candidate_features(d)
    y=d["isFraud"].astype(int).to_numpy()

    n=len(d); te=int(n*0.70); ve=int(n*0.85)
    tr=np.arange(0,te); va=np.arange(te,ve); tt=np.arange(ve,n)
    imp=SimpleImputer(strategy="median")
    xa_tr=imp.fit_transform(cand.iloc[tr]); xa_va=imp.transform(cand.iloc[va]); xa_tt=imp.transform(cand.iloc[tt])
    lo=np.nanquantile(xa_tr,CLIP_LOWER,axis=0); hi=np.nanquantile(xa_tr,CLIP_UPPER,axis=0)
    xa_tr=np.clip(xa_tr,lo,hi); xa_va=np.clip(xa_va,lo,hi); xa_tt=np.clip(xa_tt,lo,hi)
    sc=StandardScaler(); xa_tr=sc.fit_transform(xa_tr); xa_va=sc.transform(xa_va); xa_tt=sc.transform(xa_tt)
    mi_n=min(MI_SAMPLE_SIZE,len(xa_tr)); ridx=np.random.default_rng(SEED).choice(len(xa_tr),mi_n,replace=False)
    mi=mutual_info_classif(xa_tr[ridx],y[tr][ridx],random_state=SEED)
    names=list(cand.columns); mser=pd.Series(mi,index=names).sort_values(ascending=False)
    sel=list(mser.head(64).index)
    ids=[names.index(s) for s in sel]
    xa_tr=xa_tr[:,ids].astype(np.float32); xa_va=xa_va[:,ids].astype(np.float32); xa_tt=xa_tt[:,ids].astype(np.float32)
    imtr=xa_tr.reshape(-1,1,8,8); imva=xa_va.reshape(-1,1,8,8); imtt=xa_tt.reshape(-1,1,8,8)

    group=d["nameOrig"].astype(str).to_numpy()
    hidx=build_history_indices(group,SEQUENCE_LENGTH)
    full=np.zeros((len(d),64),dtype=np.float32)
    full[tr]=xa_tr; full[va]=xa_va; full[tt]=xa_tt
    seq_tr=materialize_history(hidx[tr],full,xa_tr)
    seq_va=materialize_history(hidx[va],full,xa_va)
    seq_tt=materialize_history(hidx[tt],full,xa_tt)

    ds_tr=HybridFraudDataset(imtr,seq_tr,y[tr]); ds_va=HybridFraudDataset(imva,seq_va,y[va]); ds_tt=HybridFraudDataset(imtt,seq_tt,y[tt])
    ld_tr=DataLoader(ds_tr,batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS)
    ld_va=DataLoader(ds_va,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS)
    ld_tt=DataLoader(ds_tt,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS)

    model=DFFMVSMModel(True,True,True,True)
    model,_=train_model(model,ld_tr,ld_va,f"external_{name}",epochs=min(EPOCHS,20),patience=min(PATIENCE,5))
    metrics,_,_,_,_,_=evaluate_model(model,ld_tt)
    metrics["Dataset"]=name
    return metrics


if RUN_EXTERNAL_DATASETS and RUN_PROFILE == "PAPER":
    external_results=[]
    try:
        ps=Path(EXTERNAL_PATHS["PaySim"])
        if ps.exists(): external_results.append(run_external_dataset("PaySim",load_paysim(ps)))
    except Exception as e:
        print("PaySim external run failed:",repr(e))
    try:
        eu=Path(EXTERNAL_PATHS["European Credit Card"])
        if eu.exists(): external_results.append(run_external_dataset("European Credit Card",load_european(eu)))
    except Exception as e:
        print("European external run failed:",repr(e))
    try:
        it=Path(EXTERNAL_PATHS["IEEE-CIS transaction"]); ii=Path(EXTERNAL_PATHS["IEEE-CIS identity"])
        if it.exists(): external_results.append(run_external_dataset("IEEE-CIS",load_ieee_cis(it,ii if ii.exists() else None)))
    except Exception as e:
        print("IEEE-CIS external run failed:",repr(e))
    ext_df=pd.DataFrame(external_results)
    if not ext_df.empty:
        ext_df.to_csv(OUTPUT_DIR/"cross_dataset_results_actual.csv",index=False)
        print(ext_df.to_string(index=False,float_format=lambda x:f"{x:.6f}"))
else:
    ext_df=pd.DataFrame()
    print("External dataset execution disabled. Use the paths above and set RUN_PROFILE='PAPER' if appropriate.")

In [ ]:
# ============================================================
# 38. MANUSCRIPT CROSS-DATASET REFERENCE FIGURE
# ============================================================

m=MANUSCRIPT_CROSSDATA.copy()
plt.figure(figsize=(10,5))
plt.bar(m["Dataset"],m["Accuracy_pct"])
plt.xticks(rotation=30,ha="right")
plt.ylabel("Reported Accuracy (%)")
plt.title("Manuscript-Reported Cross-Dataset Accuracy — Reference Only")
plt.grid(axis="y",alpha=0.25); plt.tight_layout()
plt.savefig(OUTPUT_DIR/"manuscript_crossdataset_reference.png",dpi=300,bbox_inches="tight")
plt.show()

## 39A. Manuscript-reported baseline/CV reference figures

These figures use only the numbers explicitly reported in the manuscript. They are references for visual comparison and are not new experimental measurements.

In [ ]:
# ============================================================
# 39A. MANUSCRIPT-REPORTED BASELINE AND CV REFERENCE FIGURES
# ============================================================

# Baseline reference
b = MANUSCRIPT_BASELINE.copy()
x = np.arange(len(b))
width = 0.25
plt.figure(figsize=(10,6))
plt.bar(x-width, b["Accuracy_pct"], width, label="Accuracy (%)")
plt.bar(x, b["Precision_pct"], width, label="Precision (%)")
plt.bar(x+width, b["Recall_pct"], width, label="Recall (%)")
plt.xticks(x, b["Model"], rotation=20, ha="right")
plt.ylabel("Reported score (%)")
plt.title("Manuscript-Reported CNN/LSTM/DFFM Baseline Comparison")
plt.legend(); plt.grid(axis="y", alpha=0.25); plt.tight_layout()
plt.savefig(OUTPUT_DIR/"manuscript_baseline_reference.png", dpi=300, bbox_inches="tight")
plt.show(); plt.close()

# 5-fold CV reference
c = MANUSCRIPT_CV.copy()
plt.figure(figsize=(9,5))
plt.plot(c["Fold"], c["Accuracy_pct"], marker="o", label="Accuracy (%)")
plt.plot(c["Fold"], c["F1"]*100, marker="s", label="F1 (%)")
plt.plot(c["Fold"], c["ROC_AUC"]*100, marker="^", label="ROC-AUC × 100")
plt.plot(c["Fold"], c["PR_AUC"]*100, marker="d", label="PR-AUC × 100")
plt.ylabel("Reported value / scaled score")
plt.title("Manuscript-Reported 5-Fold Cross-Validation")
plt.legend(); plt.grid(alpha=0.25); plt.tight_layout()
plt.savefig(OUTPUT_DIR/"manuscript_cv_reference.png", dpi=300, bbox_inches="tight")
plt.show(); plt.close()


In [ ]:
# ============================================================
# 39. MANUSCRIPT CONSISTENCY CHECKS
# ============================================================

consistency = []
# DFFM FPR discrepancy
fpr_from_spec = 1 - 0.979
consistency.append({
    "Check":"DFFM FPR paragraph vs Table IV",
    "Paragraph_FPR_pct":REPORT["reported_dffm_fpr_paragraph_pct"],
    "Table_FPR_pct":REPORT["reported_dffm_fpr_table_pct"],
    "FPR_from_reported_specificity_pct":round(fpr_from_spec*100,2),
    "Status":"INCONSISTENT" if REPORT["reported_dffm_fpr_paragraph_pct"] != REPORT["reported_dffm_fpr_table_pct"] else "CONSISTENT"
})

# CV mean/std check from manuscript fold values
cv_mean=MANUSCRIPT_CV.Accuracy_pct.mean()
cv_std=MANUSCRIPT_CV.Accuracy_pct.std(ddof=1)
consistency.append({
    "Check":"5-fold CV arithmetic from reported fold accuracies",
    "Reported_mean_pct":97.58,
    "Computed_mean_pct":round(cv_mean,4),
    "Reported_std_pct":0.27,
    "Computed_sample_std_pct":round(cv_std,4),
    "Status":"CHECK" if abs(cv_mean-97.58)>0.01 or abs(cv_std-0.27)>0.01 else "CONSISTENT"
})

consistency_df=pd.DataFrame(consistency)
consistency_df.to_csv(OUTPUT_DIR/"manuscript_consistency_checks.csv",index=False)
print(consistency_df.to_string(index=False))

In [ ]:
# ============================================================
# 40. ACTUAL-vs-REPORTED COMPARISON TABLE
# ============================================================

if not actual_ablation_df.empty:
    rep=MANUSCRIPT_ABLATION[["Model","Accuracy_pct","PR_AUC"]].copy()
    act=actual_ablation_df[["Model","Accuracy","PR_AUC"]].copy()
    act["Actual_Accuracy_pct"]=act["Accuracy"]*100
    act=act[["Model","Actual_Accuracy_pct","PR_AUC"]]
    comp=rep.merge(act,on="Model",how="outer")
    comp["Accuracy_Difference_points"]=comp["Actual_Accuracy_pct"]-comp["Accuracy_pct"]
    comp["PR_AUC_Difference"]=comp["PR_AUC_y"]-comp["PR_AUC_x"]
    comp.rename(columns={"Accuracy_pct":"Manuscript_Accuracy_pct","PR_AUC_x":"Manuscript_PR_AUC","PR_AUC_y":"Actual_PR_AUC"},inplace=True)
    comp.to_csv(OUTPUT_DIR/"actual_vs_manuscript_ablation.csv",index=False)
    display(comp)
else:
    print("Actual ablation comparison will appear after training.")

## 41A. Paper architecture schematic

A clean schematic is generated from the implementation rather than copied from the manuscript figure, so it reflects the executable notebook pipeline.

In [ ]:
# ============================================================
# 41A. GENERATE IMPLEMENTATION-ALIGNED ARCHITECTURE SCHEMATIC
# ============================================================

from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(15, 6))
ax.set_xlim(0, 15); ax.set_ylim(0, 6); ax.axis("off")

boxes = [
    (0.2,2.1,2.0,1.8,"CIFER\nTransaction\nfeatures"),
    (2.8,2.1,2.0,1.8,"Preprocess\n+ MI selection\n64 features"),
    (5.4,3.45,2.0,1.4,"ViT\nGlobal attention"),
    (5.4,2.3,2.0,1.4,"Swin\nWindow + shifted\nattention"),
    (5.4,1.15,2.0,1.4,"Mamba\n20-step history\nSelective SSM"),
    (8.2,2.1,2.2,1.8,"DFFM\nalpha, beta, gamma\nadaptive fusion"),
    (11.1,2.1,1.6,1.8,"Classifier\nFC + dropout\nsoftmax"),
    (13.25,2.1,1.4,1.8,"Fraud /\nLegitimate\n+ XAI"),
]

for x,y,w,h,label in boxes:
    patch=FancyBboxPatch((x,y),w,h,boxstyle="round,pad=0.04",linewidth=1.5,facecolor="white")
    ax.add_patch(patch)
    ax.text(x+w/2,y+h/2,label,ha="center",va="center",fontsize=10)

arrows=[
    ((2.2,3.0),(2.8,3.0)),
    ((4.8,3.0),(5.4,4.15)),
    ((4.8,3.0),(5.4,3.0)),
    ((4.8,3.0),(5.4,1.85)),
    ((7.4,4.15),(8.2,3.35)),
    ((7.4,3.0),(8.2,3.0)),
    ((7.4,1.85),(8.2,2.65)),
    ((10.4,3.0),(11.1,3.0)),
    ((12.7,3.0),(13.25,3.0)),
]
for a,b in arrows:
    ax.add_patch(FancyArrowPatch(a,b,arrowstyle="->",mutation_scale=16,linewidth=1.4))

ax.text(6.35,5.35,"8x8 tabular-to-image mapping -> parallel global, local, temporal branches",ha="center",fontsize=12,fontweight="bold")
ax.text(9.3,0.55,"DFFM: transaction-adaptive softmax fusion of ViT, Swin, and Mamba",ha="center",fontsize=11)

path=OUTPUT_DIR/"dffm_vsm_architecture_schematic.png"
plt.savefig(path,dpi=300,bbox_inches="tight")
plt.show(); plt.close()
print("Saved:",path)


## 41. Final paper-output files

At the end of the run, the notebook writes:

- `ablation_results_actual.csv` and a LaTeX-ready table
- all actual confusion matrices
- actual PR and ROC curves
- actual training/validation loss plots
- actual DFFM weights
- `selected_64_features.csv` and exact 8×8 feature mapping
- actual SHAP ranking (when SHAP completes)
- actual Grad-CAM heatmaps
- chronological leakage/overlap audit
- unseen-sender fraud audit
- optional stress-test results
- optional CNN/LSTM baseline results
- optional 5-fold chronological CV
- optional external-dataset results
- manuscript-reported reference tables/figures
- `manuscript_consistency_checks.csv`

The manuscript-reported tables and the actual experimental tables are deliberately kept separate.

In [ ]:
# ============================================================
# 42. SAVE A SINGLE PAPER-READY METRICS FILE + LATEX TABLE
# ============================================================

if not actual_ablation_df.empty:
    paper_cols=["Model","Accuracy","PR_AUC","MCC","Balanced_Accuracy","FPR","Recall","Specificity"]
    paper_df=actual_ablation_df[paper_cols].copy()
    paper_df.to_csv(OUTPUT_DIR/"paper_ready_metrics_actual.csv",index=False)

    tex_df=paper_df.copy()
    tex_df["Accuracy"]=(tex_df["Accuracy"]*100).map(lambda x:f"{x:.2f}\\%")
    tex_df["FPR"]=(tex_df["FPR"]*100).map(lambda x:f"{x:.2f}\\%")
    tex_df["Recall"]=(tex_df["Recall"]*100).map(lambda x:f"{x:.4f}\\%")
    tex_df["Specificity"]=(tex_df["Specificity"]*100).map(lambda x:f"{x:.4f}\\%")
    for c in ["PR_AUC","MCC","Balanced_Accuracy"]:
        tex_df[c]=tex_df[c].map(lambda x:f"{x:.4f}")
    latex=tex_df.to_latex(index=False,escape=False)
    with open(OUTPUT_DIR/"paper_ready_metrics_actual.tex","w") as f:
        f.write("\\begin{table}[htbp]\n\\centering\n")
        f.write("\\caption{Actual ablation results obtained by the reproducible notebook on the configured CIFER run.}\n")
        f.write("\\label{tab:actual_ablation}\n")
        f.write(latex)
        f.write("\\end{table}\n")
    print(latex)
else:
    print("No actual model results to export in this profile.")

In [ ]:
# ============================================================
# 43. FINAL RUN SUMMARY
# ============================================================

summary={
    "output_directory": str(OUTPUT_DIR),
    "profile": RUN_PROFILE,
    "num_rows_loaded": int(len(raw_df)),
    "num_selected_features": int(NUM_SELECTED_FEATURES),
    "image_shape": [1,8,8],
    "sequence_length": int(SEQUENCE_LENGTH),
    "mamba_backend": MAMBA_BACKEND,
    "official_mamba_required": bool(REQUIRE_OFFICIAL_MAMBA),
    "gan_synthetic_train_rows": int(len(X_synth)),
    "actual_ablation_completed": int(len(actual_ablation_df)),
    "baseline_completed": int(len(baseline_actual_df)),
    "cv_completed": int(len(cv_df)),
    "external_datasets_completed": int(len(ext_df)),
}

with open(OUTPUT_DIR/"final_run_summary.json","w") as f:
    json.dump(summary,f,indent=2)

print(json.dumps(summary,indent=2))

In [ ]:
# ============================================================
# 44. LIST OUTPUT FILES
# ============================================================

for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print(p.relative_to(OUTPUT_DIR))

In [ ]:
# ============================================================
# 45. ZIP ALL RESULTS FOR DOWNLOAD
# ============================================================

import shutil
zip_path = shutil.make_archive(
    "/content/DFFM_VSM_CIFER_RESULTS",
    "zip",
    OUTPUT_DIR
)
print("ZIP:", zip_path)